# 07 · SQL for Data Science — Complete Reference (runnable with sqlite3)

SQL is the *other* half of a data analyst's toolkit: gathering, cleaning and
exploring data **where it lives** — inside the database. Every statement in
this notebook runs **live** against an in-memory SQLite database
(`sqlite3.connect(":memory:")`) and prints its result with pandas, so you can
edit any query and re-run it immediately.

Covers sessions 30-37 plus the Zomato / Laptops / Flights case studies,
interviews, database design, views, transactions — in playlist order.

> **Note:** SQLite is a *library* database (no server) chosen because it ships
> with Python. Dialect differences vs MySQL/PostgreSQL are called out
> **NOTE (dialect):** at each spot that matters.

## ▶ Sessions covered

| # | Session | Video |
|---|---------|-------|
| 96 | Session 30 - Database Fundamentals | https://www.youtube.com/watch?v=CTLwYjh92xw |
| 99 | Session 31 - SQL DDL Commands | https://www.youtube.com/watch?v=ny1mh6VUpnQ |
| 101 | Session 32 - SQL DML Commands | https://www.youtube.com/watch?v=kzMZDlvHFak |
| 102 | Session 33 - SQL Grouping + Sorting | https://www.youtube.com/watch?v=nsKcmOly0UY |
| 104 | Session 34 - SQL Joins | https://www.youtube.com/watch?v=s6xAVqYdmJ4 |
| 107 | Session 35 - Subqueries in SQL | https://www.youtube.com/watch?v=YYq47MN3TZI |
| 108 | SQL Case Study 1 \| Zomato Dataset | https://www.youtube.com/watch?v=jYwcB0KEFfY |
| 111 | SQL Interview Questions Part 1 | https://www.youtube.com/watch?v=QlenVBvzrW4 |
| 112 | Session 36 - Window Functions in SQL | https://www.youtube.com/watch?v=3LJ-yWBQBGs |
| 115 | Session 37 - Window Functions Part 2 | https://www.youtube.com/watch?v=YgizUxmBFK8 |
| 116 | Session 37 (Cont..) - Window Functions Part 3 | https://www.youtube.com/watch?v=e0HQVOdfqlQ |
| 117 | Session on Data Cleaning using SQL \| Laptops | https://www.youtube.com/watch?v=NsOXJdRIamc |
| 118 | Session on EDA using SQL \| Laptops | https://www.youtube.com/watch?v=dv8TkUTlQvI |
| 124 | Session on Datetime in SQL | https://www.youtube.com/watch?v=OY9YiALhSR4 |
| 127 | SQL Datetime Case Study on Flights | https://www.youtube.com/watch?v=zY1UCTko64E |
| 128 | Session on Database Design \| Normalization | https://www.youtube.com/watch?v=8SOs0URTvr0 |
| 132 | Session on Views and UDFs in SQL | https://www.youtube.com/watch?v=dMoKTx1bVnY |
| 133 | Session on Transactions & Stored Procedures | https://www.youtube.com/watch?v=-O_9iR7XsJw |

In [1]:
import sqlite3
import pandas as pd

print("SQLite version:", sqlite3.sqlite_version)

con = sqlite3.connect(":memory:")     # in-memory DB — lives only for this kernel
con.isolation_level = None            # autocommit: WE drive BEGIN/COMMIT (§12)


def q(sql: str, params=None) -> pd.DataFrame:
    """Run a SQL statement and return the result as a DataFrame."""
    return pd.read_sql_query(sql, con, params=params)


# --- schema (DDL) -------------------------------------------------------
con.executescript("""
CREATE TABLE departments (
    dept_id   INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL UNIQUE,
    location  TEXT
);

CREATE TABLE employees (
    emp_id     INTEGER PRIMARY KEY,
    emp_name   TEXT NOT NULL,
    dept_id    INTEGER NOT NULL REFERENCES departments(dept_id),
    manager_id INTEGER REFERENCES employees(emp_id),   -- self reference
    salary     REAL NOT NULL CHECK(salary > 0),
    bonus      REAL,                                    -- NULLs on purpose
    hire_date  TEXT NOT NULL,                           -- ISO 'YYYY-MM-DD'
    age        INTEGER,
    city       TEXT
);

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    cust_name   TEXT NOT NULL,
    city        TEXT,
    signup_date TEXT,
    tier        TEXT DEFAULT 'Bronze'
);

CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
    order_date  TEXT NOT NULL,
    amount      REAL NOT NULL CHECK(amount >= 0),
    status      TEXT DEFAULT 'completed'
);
""")

# --- seed data (DML) ----------------------------------------------------
con.executemany("INSERT INTO departments VALUES(?, ?, ?)", [
    (1, "HR", "Mumbai"), (2, "IT", "Bengaluru"), (3, "Finance", "Delhi"),
    (4, "Sales", "Pune"), (5, "Marketing", "Mumbai"), (6, "Support", "Hyderabad"),
    (7, "Legal", "Chennai"),          # Legal has NO employees (for join demos)
])

con.executemany(
    "INSERT INTO employees VALUES(?, ?, ?, ?, ?, ?, ?, ?, ?)", [
        (1, "Rahul", 2, None, 90000, 5000, "2019-03-15", 32, "Bengaluru"),
        (2, "Priya", 2, 1, 75000, None, "2020-07-01", 28, "Bengaluru"),
        (3, "Amit", 1, None, 62000, 3000, "2017-11-20", 40, "Mumbai"),
        (4, "Sneha", 1, 3, 58000, None, "2021-01-10", 26, "Mumbai"),
        (5, "Vikram", 3, None, 88000, 8000, "2016-05-05", 45, "Delhi"),
        (6, "Neha", 3, 5, 70000, None, "2022-09-12", 30, "Delhi"),
        (7, "Arjun", 4, None, 66000, 4000, "2018-02-18", 35, "Pune"),
        (8, "Kavya", 4, 7, 54000, None, "2023-04-01", 25, "Pune"),
        (9, "Rohan", 2, 1, 90000, 6000, "2021-06-25", 29, "Bengaluru"),
        (10, "Divya", 5, None, 60000, None, "2019-12-30", 33, "Mumbai"),
        (11, "Manish", 6, None, 48000, 2000, "2020-02-14", 31, "Hyderabad"),
        (12, "Pooja", 6, 11, 46000, None, "2024-01-08", 24, "Hyderabad"),
    ])

con.executemany("INSERT INTO customers VALUES(?, ?, ?, ?, ?)", [
    (1, "Amita", "Delhi", "2024-01-02", "Gold"),
    (2, "Sunil", "Mumbai", "2024-01-15", "Silver"),
    (3, "Meera", "Pune", "2024-02-01", "Gold"),
    (4, "Rakesh", "Delhi", "2024-02-10", "Bronze"),
    (5, "Zoya", "Kolkata", "2024-02-14", "Silver"),
    (6, "Karthik", "Chennai", "2024-03-01", "Bronze"),
    (7, "Nisha", "Mumbai", "2024-03-05", "Gold"),
    (8, "Vivek", "Pune", "2024-03-11", "Silver"),
    (9, "Tara", "Delhi", "2024-03-18", "Bronze"),     # places NO orders
    (10, "Imran", "Hyderabad", "2024-03-25", "Bronze"),  # places NO orders
])

con.executemany("INSERT INTO orders VALUES(?, ?, ?, ?, ?)", [
    (1, 1, "2024-01-05", 1200, "completed"),
    (2, 1, "2024-01-06", 800, "completed"),
    (3, 1, "2024-01-20", 1500, "cancelled"),
    (4, 2, "2024-01-06", 2200, "completed"),
    (5, 2, "2024-02-03", 900, "completed"),
    (6, 3, "2024-02-05", 3000, "completed"),
    (7, 3, "2024-02-06", 450, "completed"),
    (8, 3, "2024-02-07", 700, "pending"),
    (9, 4, "2024-02-14", 1800, "completed"),
    (10, 5, "2024-02-14", 600, "cancelled"),
    (11, 5, "2024-03-02", 2500, "completed"),
    (12, 6, "2024-03-08", 1100, "completed"),
    (13, 7, "2024-03-15", 950, "completed"),
    (14, 7, "2024-03-16", 1300, "completed"),
    (15, 8, "2024-03-21", 2000, "completed"),
])

# --- look at what we built ---------------------------------------------
print("tables:", q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")["name"].tolist())
for t in ("departments", "employees", "customers", "orders"):
    df = q(f"SELECT * FROM {t}")
    print(f"\n--- {t}  ({len(df)} rows) ---")
    print(df)


SQLite version: 3.35.5
tables: ['customers', 'departments', 'employees', 'orders']

--- departments  (7 rows) ---
   dept_id  dept_name   location
0        1         HR     Mumbai
1        2         IT  Bengaluru
2        3    Finance      Delhi
3        4      Sales       Pune
4        5  Marketing     Mumbai
5        6    Support  Hyderabad
6        7      Legal    Chennai

--- employees  (12 rows) ---
    emp_id emp_name  dept_id  manager_id   salary   bonus   hire_date  age  \
0        1    Rahul        2         NaN  90000.0  5000.0  2019-03-15   32   
1        2    Priya        2         1.0  75000.0     NaN  2020-07-01   28   
2        3     Amit        1         NaN  62000.0  3000.0  2017-11-20   40   
3        4    Sneha        1         3.0  58000.0     NaN  2021-01-10   26   
4        5   Vikram        3         NaN  88000.0  8000.0  2016-05-05   45   
5        6     Neha        3         5.0  70000.0     NaN  2022-09-12   30   
6        7    Arjun        4         NaN  6600

---
## 1. Database fundamentals

▶ Session 30 - Database Fundamentals: https://www.youtube.com/watch?v=CTLwYjh92xw

### DBMS vs RDBMS

| | **DBMS** | **RDBMS** |
|---|---|---|
| Storage model | files, documents, graphs, key-value | **tables** with rows & columns |
| Relations | none required | foreign keys enforce relations |
| Query language | varies | SQL |
| Examples | MongoDB, Redis, Neo4j | PostgreSQL, MySQL, SQLite, Oracle |

### Table / row / column / keys

```text
 departments                       employees
+---------+----------+---------+   +--------+---------+---------+------------+
| dept_id | dept_name| location|   | emp_id | emp_name| dept_id | manager_id |
+---------+----------+---------+   +--------+---------+---------+------------+
| 1       | HR       | Mumbai  |   | 1      | Rahul   | 2       | NULL  <─────┐
| 2       | IT       | Bengaluru   | 2      | Priya   | 2       | 1  ─────────┘
+---------+----------+---------+   +--------+---------+---------+
   PRIMARY KEY ^   UNIQUE ^            PK ^       FK to departments ^
                                       FK to *same* table (self reference) ^
```

- **Primary key (PK)** — uniquely identifies a row; never NULL.
- **Foreign key (FK)** — a PK of *another* table, pointing at a parent row.
- **Composite key** — a key made of 2+ columns.

### Keys

| Key | Definition | In our schema |
|---|---|---|
| candidate | any column (set) that *could* be the PK | `emp_name` (unique here) |
| primary | the chosen candidate key | `employees.emp_id` |
| unique | allows NULL, forbids repeats | `departments.dept_name` |
| foreign | references a PK of another (or same) table | `employees.dept_id` |
| composite | multi-column key | `(order_id, product_id)` style |

### ER diagram concepts (entity-relationship)

- **Entity** — a noun → table (`customer`, `order`); **attribute** → column.
- **Relationship** — a verb between entities: `customer --places--> order`.
- **Cardinality**:
  - **1:1** — one user has one profile
  - **1:N** — one department has many employees ← our `departments/employees`
  - **M:N** — students ↔ courses (needs a *junction* table `enrollments`)
- **Participation** — total (every row must match) vs optional (may not).
- **Degree** — arity of the relationship (binary is the common case).

### OLTP vs OLAP

| | **OLTP** (operations) | **OLAP** (analytics) |
|---|---|---|
| Purpose | run the business | understand the business |
| Queries | short point reads (`WHERE id=?`) | huge scans, aggregates |
| Writes | thousands/sec, small | batch loads, rarely updated |
| Schema | normalized (§13) | star/snowflake, denormalized |
| Rows touched | 1-100 | millions |
| Example | place an order | "revenue by month by city" |

### SQLite vs MySQL / PostgreSQL

| Feature | SQLite | MySQL | PostgreSQL |
|---|---|---|---|
| architecture | in-process file | client/server | client/server |
| concurrency | 1 writer | good | excellent |
| data types | dynamic (types per column are hints) | rigid | strict, rich types |
| JSON support | `json_each()` | good | excellent |
| window functions | ✅ (3.25+) | ✅ (8.0+) | ✅ |
| `RIGHT/FULL OUTER JOIN` | **only 3.39+** (see §7) | ✅ | ✅ |
| `TRUNCATE` | ❌ (use `DELETE FROM`) | ✅ | ✅ |
| stored procedures | ❌ | ✅ (SQL/PSM) | ✅ (PL/pgSQL) |
| `STDDEV()` | ❌ (compute it, §15) | ✅ | ✅ |
| `GROUP BY GROUPING SETS` | build-dependent (❌ here) | ✅ | ✅ |
| `ANY/ALL` subqueries | ❌ (rewrite with IN/MAX) | ✅ | ✅ |
| `AUTOINCREMENT` | `INTEGER PRIMARY KEY` auto-rowid | `AUTOINCREMENT` | `SERIAL` |
| best for | learning, embedded, tests | web at scale | analytics + correctness |

In [2]:
print("SQLite:", sqlite3.sqlite_version)
print(q("PRAGMA table_info(employees)"))         # columns, types, PK
print(q("PRAGMA foreign_key_list(employees)"))    # FKs declared on the table
print(q("SELECT * FROM pragma_index_list('employees')"))   # auto indexes (none yet)


SQLite: 3.35.5
   cid        name     type  notnull dflt_value  pk
0    0      emp_id  INTEGER        0       None   1
1    1    emp_name     TEXT        1       None   0
2    2     dept_id  INTEGER        1       None   0
3    3  manager_id  INTEGER        0       None   0
4    4      salary     REAL        1       None   0
5    5       bonus     REAL        0       None   0
6    6   hire_date     TEXT        1       None   0
7    7         age  INTEGER        0       None   0
8    8        city     TEXT        0       None   0
   id  seq        table        from       to  on_update  on_delete match
0   0    0    employees  manager_id   emp_id  NO ACTION  NO ACTION  NONE
1   1    0  departments     dept_id  dept_id  NO ACTION  NO ACTION  NONE
Empty DataFrame
Columns: [seq, name, unique, origin, partial]
Index: []


**Section notes — fundamentals:**

| Concept | One-liner |
|---|---|
| DBMS / RDBMS | data platform / relational (tables + SQL) |
| PK | unique, not null identity of a row |
| FK | pointer to a parent PK — enforces referential integrity |
| 1:N | one-to-many: put the FK on the "many" side |
| M:N | needs a junction table |
| OLTP vs OLAP | write-fast operational vs read-fast analytical |
| SQLite here | no server; 3.35.5 → emulate RIGHT/FULL joins (§7) |

---
## 2. DDL — Data Definition Language

▶ Session 31 - SQL DDL Commands: https://www.youtube.com/watch?v=ny1mh6VUpnQ

DDL defines/changes **structure**: `CREATE`, `ALTER`, `DROP`, `TRUNCATE`, `RENAME`.
It is auto-committed in most RDBMS (and, with our `isolation_level=None`, here too).

In [3]:
# CREATE TABLE with every common constraint
con.executescript("""
CREATE TABLE IF NOT EXISTS products (
    product_id  INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL UNIQUE,
    price       REAL NOT NULL DEFAULT 9.99 CHECK(price > 0),
    dept_id     INTEGER REFERENCES departments(dept_id),
    launched    TEXT
);
""")

q1 = "INSERT INTO products(product_id, product_name, price, dept_id, launched) VALUES(?, ?, ?, ?, ?)"
con.execute(q1, (1, "Notebook", 60.0, 4, "2023-06-01"))
con.execute(q1, (2, "Marker", 25.5, 4, "2023-06-01"))
con.execute(q1, (3, "Stapler", 180.0, 6, "2024-01-15"))
print(q("SELECT * FROM products"))


   product_id product_name  price  dept_id    launched
0           1     Notebook   60.0        4  2023-06-01
1           2       Marker   25.5        4  2023-06-01
2           3      Stapler  180.0        6  2024-01-15


In [4]:
# constraints reject bad data — Python sees sqlite3.IntegrityError
violations = [
    ("duplicate PK", "INSERT INTO products VALUES(1, 'Pen', 10, 4, '2024-01-01')"),
    ("NULL in NOT NULL", "INSERT INTO products(product_id, product_name) VALUES(4, NULL)"),
    ("CHECK price > 0", "INSERT INTO products VALUES(4, 'Broken', -5, 4, '2024-01-01')"),
    ("bad FK (dept 99)", "INSERT INTO products VALUES(4, 'Ghost', 10, 99, '2024-01-01')"),
]
for label, sql in violations:
    try:
        con.execute(sql)
        print(f"{label}: NO ERROR (unexpected!)")
    except sqlite3.IntegrityError as e:
        print(f"{label:20s} -> IntegrityError: {e}")
print("\nstill only the 3 valid rows:", len(q("SELECT * FROM products")))


duplicate PK         -> IntegrityError: UNIQUE constraint failed: products.product_id
NULL in NOT NULL     -> IntegrityError: NOT NULL constraint failed: products.product_name
CHECK price > 0      -> IntegrityError: CHECK constraint failed: price > 0
bad FK (dept 99): NO ERROR (unexpected!)

still only the 3 valid rows: 4


In [5]:
# ### ALTER TABLE — ADD / DROP / RENAME COLUMN
con.execute("ALTER TABLE products ADD COLUMN weight_g REAL DEFAULT 0")
con.execute("UPDATE products SET weight_g = 70 WHERE product_id = 1")
con.execute("ALTER TABLE products RENAME COLUMN weight_g TO weight_grams")
print(q("SELECT * FROM products"))

con.execute("ALTER TABLE products DROP COLUMN launched")     # SQLite >= 3.35
print("after DROP COLUMN:", q("PRAGMA table_info(products)")["name"].tolist())


   product_id product_name  price  dept_id    launched  weight_grams
0           1     Notebook   60.0        4  2023-06-01          70.0
1           2       Marker   25.5        4  2023-06-01           0.0
2           3      Stapler  180.0        6  2024-01-15           0.0
3           4        Ghost   10.0       99  2024-01-01           0.0
after DROP COLUMN: ['product_id', 'product_name', 'price', 'dept_id', 'weight_grams']


In [6]:
# ### DROP vs TRUNCATE vs DELETE — removing rows/table
con.execute("CREATE TABLE scratch(x INTEGER)")   # disposable table
con.executemany("INSERT INTO scratch VALUES(?)", [(i,) for i in range(100)])

print("before          :", len(q("SELECT * FROM scratch")))
con.execute("DELETE FROM scratch")               # TRUNCATE in SQLite (no TRUNCATE stmt!)
print("after DELETE all:", len(q("SELECT * FROM scratch")))

con.execute("DROP TABLE scratch")                # removes the table itself
print("tables now:",
      q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")["name"].tolist())


before          : 100
after DELETE all: 0
tables now: ['customers', 'departments', 'employees', 'orders', 'products']


In [7]:
# RENAME a table (SQLite: ALTER TABLE old RENAME TO new)
con.execute("CREATE TABLE old_reports(id INTEGER PRIMARY KEY, title TEXT)")
con.execute("ALTER TABLE old_reports RENAME TO reports")
print(q("SELECT name FROM sqlite_master WHERE type='table' AND name LIKE '%report%'"))
con.execute("DROP TABLE reports")


      name
0  reports


### DDL cheat-sheet

| Task | SQLite | MySQL / PostgreSQL |
|---|---|---|
| add column | `ALTER TABLE t ADD COLUMN c TYPE` | same |
| drop column | `ALTER TABLE t DROP COLUMN c` (3.35+) | same / `DROP COLUMN` |
| rename column | `ALTER TABLE t RENAME COLUMN a TO b` | MySQL: `CHANGE`/`MODIFY` |
| rename table | `ALTER TABLE a RENAME TO b` | `RENAME TABLE a TO b` (MySQL) |
| remove all rows | `DELETE FROM t` (**no TRUNCATE**) | `TRUNCATE t` (faster, DDL) |
| remove table | `DROP TABLE t` (structure + data) | same |

> **Gotcha:** `DELETE` is DML (row-by-row, logged, where-clause friendly, can
> roll back); `TRUNCATE` is DDL (whole table, resets identity, usually not
> rollback-able). SQLite only has the former.

**Section notes — DDL:** `CREATE TABLE (...constraints...)` · `ALTER TABLE
ADD/DROP/RENAME COLUMN` · `ALTER TABLE ... RENAME TO ...` · `DROP TABLE` ·
`DELETE FROM` (truncate substitute) · constraints: `PRIMARY KEY`, `NOT NULL`,
`UNIQUE`, `DEFAULT`, `CHECK`, `REFERENCES`.

---
## 3. DML — Data Manipulation Language

▶ Session 32 - SQL DML Commands: https://www.youtube.com/watch?v=kzMZDlvHFak

DML changes **data**: `INSERT`, `UPDATE`, `DELETE` (+ `MERGE`/upsert in other dialects).

In [8]:
# INSERT — single row, multi row, and INSERT ... SELECT
con.executescript("CREATE TABLE staging(id INTEGER PRIMARY KEY, tag TEXT);")
con.execute("INSERT INTO staging VALUES(10, 'solo')")                       # single
con.executemany("INSERT INTO staging VALUES(?, ?)",                         # multi (driver)
                [(11, 'batch1'), (12, 'batch2')])
con.execute("INSERT INTO staging(id, tag) VALUES(13, 'sql-multi'), (14, 'sql-multi')")  # multi SQL
print(q("SELECT * FROM staging"))

# INSERT ... SELECT: copy a projection of an existing table
con.execute("INSERT INTO staging(id, tag) SELECT emp_id, 'emp:' || emp_name FROM employees WHERE dept_id = 1")
print("\nafter INSERT ... SELECT:\n", q("SELECT * FROM staging ORDER BY id"))


   id        tag
0  10       solo
1  11     batch1
2  12     batch2
3  13  sql-multi
4  14  sql-multi

after INSERT ... SELECT:
    id        tag
0   3   emp:Amit
1   4  emp:Sneha
2  10       solo
3  11     batch1
4  12     batch2
5  13  sql-multi
6  14  sql-multi


In [9]:
# ### UPDATE — the missing-WHERE gotcha
con.execute("CREATE TABLE emp_copy AS SELECT * FROM employees")

before = q("SELECT COUNT(*) AS n, SUM(salary) AS total FROM emp_copy")
con.execute("UPDATE emp_copy SET salary = salary + 1000")            # NO WHERE!
after = q("SELECT COUNT(*) AS n, SUM(salary) AS total FROM emp_copy")
print("without WHERE -> every row changed:\n", before, "\n", after)
# NOTE: this is the classic production accident — ALWAYS write the WHERE first.

con.execute("UPDATE emp_copy SET salary = 60000 WHERE emp_id = 4")   # targeted
print("\nspecific row updated:\n", q("SELECT emp_id, emp_name, salary FROM emp_copy WHERE emp_id = 4"))

con.execute("UPDATE emp_copy SET salary = salary * 1.10 WHERE dept_id = 2 AND salary >= 75000")
print("\nmulti-condition update:\n",
      q("SELECT emp_name, salary FROM emp_copy WHERE dept_id = 2 ORDER BY salary DESC"))


without WHERE -> every row changed:
     n     total
0  12  807000.0 
     n     total
0  12  819000.0

specific row updated:
    emp_id emp_name   salary
0       4    Sneha  60000.0

multi-condition update:
   emp_name    salary
0    Rahul  100100.0
1    Rohan  100100.0
2    Priya   83600.0


In [10]:
# ### DELETE — always with a WHERE
con.execute("DELETE FROM emp_copy WHERE salary < 50000")
print("rows left after DELETE WHERE salary < 50000:", len(q("SELECT * FROM emp_copy")))


rows left after DELETE WHERE salary < 50000: 10


In [11]:
# ### UPSERT — INSERT ... ON CONFLICT (SQLite) / MERGE (other RDBMS)
try:
    con.execute("INSERT INTO products VALUES(1, 'Notebook XL', 75.0, 4, 80)")
except sqlite3.IntegrityError as e:
    print("plain INSERT on existing PK ->", e)

con.execute("""
    INSERT INTO products(product_id, product_name, price, dept_id, weight_grams)
    VALUES(1, 'Notebook XL', 75.0, 4, 80)
    ON CONFLICT(product_id) DO UPDATE SET
        price = excluded.price,
        weight_grams = excluded.weight_grams
""")
print(q("SELECT * FROM products WHERE product_id = 1"))

# DELETE-if-exists + INSERT, the pre-upsert pattern, in one statement:
con.execute("INSERT OR REPLACE INTO products VALUES(2, 'Marker Pro', 30.0, 4, 45)")
print("\nafter INSERT OR REPLACE:\n", q("SELECT * FROM products ORDER BY product_id"))


plain INSERT on existing PK -> UNIQUE constraint failed: products.product_id
   product_id product_name  price  dept_id  weight_grams
0           1     Notebook   75.0        4          80.0

after INSERT OR REPLACE:
    product_id product_name  price  dept_id  weight_grams
0           1     Notebook   75.0        4          80.0
1           2   Marker Pro   30.0        4          45.0
2           3      Stapler  180.0        6           0.0
3           4        Ghost   10.0       99           0.0


### DELETE vs TRUNCATE vs DROP

| | `DELETE` | `TRUNCATE` | `DROP` |
|---|---|---|---|
| removes | matching **rows** | **all rows** | table **+ structure** |
| `WHERE` | ✅ | ❌ | ❌ |
| speed (big table) | slow (logged row-by-row) | fast | fastest |
| roll back | yes | usually no | no |
| triggers | fire | usually no | n/a |
| SQLite | ✅ | ❌ → use `DELETE FROM` | ✅ |

**NOTE (dialect):** other engines implement `MERGE INTO t ... WHEN MATCHED THEN
UPDATE / WHEN NOT MATCHED THEN INSERT`; SQLite's equivalents are
`ON CONFLICT ... DO UPDATE` and `INSERT OR REPLACE`.

**Section notes — DML:** `INSERT [VALUES | ...SELECT]` · `UPDATE ... WHERE` ·
`DELETE ... WHERE` · upsert: `ON CONFLICT` / `INSERT OR REPLACE` ·
> **Gotcha:** no `WHERE` on UPDATE/DELETE = rewrite of the whole table.

---
## 4. SELECT basics

▶ Session 32 - SQL DML Commands: https://www.youtube.com/watch?v=kzMZDlvHFak
▶ Session 33 - SQL Grouping + Sorting: https://www.youtube.com/watch?v=nsKcmOly0UY

```text
SELECT <columns/expressions>   -- what to show
FROM <table>                   -- where the data is
WHERE <filter>                 -- row filter (before grouping)
GROUP BY <columns>             -- collapse rows into groups
HAVING <filter on groups>      -- group filter (after grouping)
ORDER BY <columns> [ASC|DESC]  -- sort the result
LIMIT n OFFSET m               -- keep only n rows, skip m
```

In [12]:
# projection, aliases, DISTINCT, expressions
print(q("SELECT emp_name, salary * 12 AS annual_ctc FROM employees WHERE salary > 70000"))
print("\nDISTINCT cities:", q("SELECT DISTINCT city FROM employees ORDER BY city")["city"].tolist())
print("DISTINCT depts  :", q("SELECT DISTINCT dept_id FROM employees ORDER BY dept_id")["dept_id"].tolist())


  emp_name  annual_ctc
0    Rahul   1080000.0
1    Priya    900000.0
2   Vikram   1056000.0
3    Rohan   1080000.0

DISTINCT cities: ['Bengaluru', 'Delhi', 'Hyderabad', 'Mumbai', 'Pune']
DISTINCT depts  : [1, 2, 3, 4, 5, 6]


In [13]:
# WHERE operators: = != <> > < >= <= , BETWEEN , IN , LIKE , IS NULL , AND/OR/NOT
print("salary > 70000            :\n", q("SELECT emp_name, salary FROM employees WHERE salary > 70000"))
print("BETWEEN 60000 AND 80000   :\n", q("SELECT emp_name, salary FROM employees WHERE salary BETWEEN 60000 AND 80000 ORDER BY salary"))
print("IN (HR, IT)               :\n", q("""
    SELECT e.emp_name, d.dept_name FROM employees e
    JOIN departments d ON e.dept_id = d.dept_id
    WHERE d.dept_name IN ('HR', 'IT') ORDER BY d.dept_name, e.emp_name"""))


salary > 70000            :
   emp_name   salary
0    Rahul  90000.0
1    Priya  75000.0
2   Vikram  88000.0
3    Rohan  90000.0
BETWEEN 60000 AND 80000   :
   emp_name   salary
0    Divya  60000.0
1     Amit  62000.0
2    Arjun  66000.0
3     Neha  70000.0
4    Priya  75000.0
IN (HR, IT)               :
   emp_name dept_name
0     Amit        HR
1    Sneha        HR
2    Priya        IT
3    Rahul        IT
4    Rohan        IT


In [14]:
# LIKE: % = any run of chars, _ = exactly one char
print(q("SELECT emp_name FROM employees WHERE emp_name LIKE 'A%'"))     # starts with A
print(q("SELECT emp_name FROM employees WHERE emp_name LIKE '%a%'"))    # contains a
print(q("SELECT emp_name FROM employees WHERE emp_name LIKE '_i%'"))    # 2nd char is i
print("escape gotcha (names with _):", q("SELECT COUNT(*) AS n FROM employees WHERE emp_name LIKE '%'")["n"][0])


  emp_name
0     Amit
1    Arjun
   emp_name
0     Rahul
1     Priya
2      Amit
3     Sneha
4    Vikram
5      Neha
6     Arjun
7     Kavya
8     Rohan
9     Divya
10   Manish
11    Pooja
  emp_name
0   Vikram
1    Divya
escape gotcha (names with _): 12


In [15]:
# NULL handling — NULL is UNKNOWN, not 0 or ''
print("bonus IS NULL    :\n", q("SELECT emp_name, bonus FROM employees WHERE bonus IS NULL"))
print("bonus IS NOT NULL:\n", q("SELECT emp_name, bonus FROM employees WHERE bonus IS NOT NULL"))
print("\nNULL comparisons are UNKNOWN (never TRUE):")
print(q("SELECT 1 = NULL AS eq, NULL = NULL AS null_eq, NULL IS NULL AS is_null"))


bonus IS NULL    :
   emp_name bonus
0    Priya  None
1    Sneha  None
2     Neha  None
3    Kavya  None
4    Divya  None
5    Pooja  None
bonus IS NOT NULL:
   emp_name   bonus
0    Rahul  5000.0
1     Amit  3000.0
2   Vikram  8000.0
3    Arjun  4000.0
4    Rohan  6000.0
5   Manish  2000.0

NULL comparisons are UNKNOWN (never TRUE):
     eq null_eq  is_null
0  None    None        1


In [16]:
# ORDER BY — multiple keys, DESC, expressions; NULL placement
print(q("SELECT emp_name, dept_id, salary FROM employees ORDER BY dept_id ASC, salary DESC"))
print("\nORDER BY expression (name length):",
      q("SELECT emp_name, LENGTH(emp_name) AS len FROM employees ORDER BY len DESC LIMIT 3"))
# NOTE (dialect): SQLite puts NULLs FIRST in ASC; emulate NULLS LAST (PostgreSQL/Oracle style):
print("\nbonus ASC with NULLS LAST emulation:\n",
      q("SELECT emp_name, bonus FROM employees ORDER BY (bonus IS NULL), bonus ASC"))


   emp_name  dept_id   salary
0      Amit        1  62000.0
1     Sneha        1  58000.0
2     Rahul        2  90000.0
3     Rohan        2  90000.0
4     Priya        2  75000.0
5    Vikram        3  88000.0
6      Neha        3  70000.0
7     Arjun        4  66000.0
8     Kavya        4  54000.0
9     Divya        5  60000.0
10   Manish        6  48000.0
11    Pooja        6  46000.0

ORDER BY expression (name length):   emp_name  len
0   Vikram    6
1   Manish    6
2    Rahul    5

bonus ASC with NULLS LAST emulation:
    emp_name   bonus
0    Manish  2000.0
1      Amit  3000.0
2     Arjun  4000.0
3     Rahul  5000.0
4     Rohan  6000.0
5    Vikram  8000.0
6     Priya     NaN
7     Sneha     NaN
8      Neha     NaN
9     Kavya     NaN
10    Divya     NaN
11    Pooja     NaN


In [17]:
# LIMIT / OFFSET — pagination (LIMIT 5 OFFSET 5 = page 2)
print(q("SELECT emp_id, emp_name FROM employees ORDER BY emp_id LIMIT 5"))
print("\npage 2:\n", q("SELECT emp_id, emp_name FROM employees ORDER BY emp_id LIMIT 5 OFFSET 5"))


   emp_id emp_name
0       1    Rahul
1       2    Priya
2       3     Amit
3       4    Sneha
4       5   Vikram

page 2:
    emp_id emp_name
0       6     Neha
1       7    Arjun
2       8    Kavya
3       9    Rohan
4      10    Divya


In [18]:
# CASE WHEN — vectorised if/else inside SQL
print(q("""
    SELECT emp_name, salary,
           CASE
             WHEN salary >= 85000 THEN 'Senior'
             WHEN salary >= 60000 THEN 'Mid'
             ELSE 'Junior'
           END AS band,
           CASE WHEN bonus IS NULL THEN 'no bonus' ELSE 'has bonus' END AS bonus_flag
    FROM employees
    ORDER BY salary DESC
"""))


   emp_name   salary    band bonus_flag
0     Rahul  90000.0  Senior  has bonus
1     Rohan  90000.0  Senior  has bonus
2    Vikram  88000.0  Senior  has bonus
3     Priya  75000.0     Mid   no bonus
4      Neha  70000.0     Mid   no bonus
5     Arjun  66000.0     Mid  has bonus
6      Amit  62000.0     Mid  has bonus
7     Divya  60000.0     Mid   no bonus
8     Sneha  58000.0  Junior   no bonus
9     Kavya  54000.0  Junior   no bonus
10   Manish  48000.0  Junior  has bonus
11    Pooja  46000.0  Junior   no bonus


**Section notes — SELECT:**

| Task | Syntax |
|---|---|
| filter | `WHERE cond [AND/OR/NOT cond]` |
| ranges | `BETWEEN a AND b` |
| sets | `IN (v1, v2)` |
| patterns | `LIKE 'a%_'` |
| nulls | `IS NULL` / `IS NOT NULL` |
| sort | `ORDER BY col [DESC], col2` |
| page | `LIMIT n OFFSET m` |
| if/else | `CASE WHEN ... THEN ... ELSE ... END` |

> **Gotcha:** `!=` and `<>` are both "not equal" (standard is `<>`).
> `WHERE col = NULL` **never** matches — always `IS NULL`.

---
## 5. Aggregation & grouping

▶ Session 33 - SQL Grouping + Sorting: https://www.youtube.com/watch?v=nsKcmOly0UY

In [19]:
# aggregate functions
print(q("""
    SELECT COUNT(*)        AS n_rows,
           COUNT(bonus)    AS n_with_bonus,        -- ignores NULLs!
           COUNT(DISTINCT city) AS n_cities,
           MIN(salary)     AS min_sal,
           MAX(salary)     AS max_sal,
           ROUND(AVG(salary), 1) AS avg_sal,
           SUM(salary)     AS total_sal
    FROM employees
"""))
# COUNT(*) counts rows · COUNT(col) counts non-NULL col · COUNT(DISTINCT col) counts distinct non-NULL


   n_rows  n_with_bonus  n_cities  min_sal  max_sal  avg_sal  total_sal
0      12             6         5  46000.0  90000.0  67250.0   807000.0


In [20]:
# GROUP BY — one row per group
print(q("""
    SELECT dept_id, COUNT(*) AS headcount, ROUND(AVG(salary), 0) AS avg_salary
    FROM employees
    GROUP BY dept_id
    ORDER BY dept_id
"""))

print("\ngroup by 2 columns (dept × city):\n", q("""
    SELECT dept_id, city, COUNT(*) AS n, SUM(salary) AS total
    FROM employees
    GROUP BY dept_id, city
    ORDER BY dept_id, city
"""))


   dept_id  headcount  avg_salary
0        1          2     60000.0
1        2          3     85000.0
2        3          2     79000.0
3        4          2     60000.0
4        5          1     60000.0
5        6          2     47000.0

group by 2 columns (dept × city):
    dept_id       city  n     total
0        1     Mumbai  2  120000.0
1        2  Bengaluru  3  255000.0
2        3      Delhi  2  158000.0
3        4       Pune  2  120000.0
4        5     Mumbai  1   60000.0
5        6  Hyderabad  2   94000.0


In [21]:
# ### HAVING vs WHERE
print("WHERE  filters ROWS before grouping:\n",
      q("SELECT emp_name FROM employees WHERE salary > 70000"))

print("\nHAVING filters GROUPS after aggregation:\n", q("""
    SELECT dept_id, COUNT(*) AS n, ROUND(AVG(salary)) AS avg_sal
    FROM employees
    GROUP BY dept_id
    HAVING COUNT(*) >= 2 AND AVG(salary) > 60000
    ORDER BY dept_id
"""))


WHERE  filters ROWS before grouping:
   emp_name
0    Rahul
1    Priya
2   Vikram
3    Rohan

HAVING filters GROUPS after aggregation:
    dept_id  n  avg_sal
0        2  3  85000.0
1        3  2  79000.0


In [22]:
# ### Execution order — why HAVING exists
# ```text
# FROM  →  WHERE  →  GROUP BY  →  HAVING  →  SELECT  →  ORDER BY  →  LIMIT
#  rows     pre-       collapse    filter     project     sort        cut
#           filter     to groups   groups     columns
# ```
# `SELECT` runs *after* grouping — that's why you cannot use a column alias in
# `WHERE`/`HAVING` (the alias doesn't exist yet), but you CAN in `ORDER BY`.


In [23]:
# GROUPING SETS / ROLLUP: subtotals in one scan (supported in PG/MySQL & some
# SQLite builds — NOT in this one, so emulate with UNION ALL):
try:
    print(q("SELECT dept_id, SUM(salary) FROM employees GROUP BY GROUPING SETS((dept_id), ())"))
except Exception as e:
    print("GROUPING SETS unsupported here ->", e)
    print("manual rollup emulation:\n", q("""
        SELECT CAST(dept_id AS TEXT) AS grp, SUM(salary) AS total FROM employees GROUP BY dept_id
        UNION ALL
        SELECT 'ALL', SUM(salary) FROM employees
        ORDER BY grp
    """))


GROUPING SETS unsupported here -> Execution failed on sql 'SELECT dept_id, SUM(salary) FROM employees GROUP BY GROUPING SETS((dept_id), ())': near "SETS": syntax error
manual rollup emulation:
    grp     total
0    1  120000.0
1    2  255000.0
2    3  158000.0
3    4  120000.0
4    5   60000.0
5    6   94000.0
6  ALL  807000.0


In [24]:
# COUNT(*) vs COUNT(col) vs COUNT(DISTINCT col) — with real NULLs in the data
print(q("""
    SELECT COUNT(*)            AS rows_total,
           COUNT(bonus)        AS non_null_bonus,
           COUNT(*) - COUNT(bonus) AS null_bonus,
           COUNT(DISTINCT city) AS distinct_cities
    FROM employees
"""))


   rows_total  non_null_bonus  null_bonus  distinct_cities
0          12               6           6                5


**Section notes — aggregation:**

| Function | Ignores NULLs? | Notes |
|---|---|---|
| `COUNT(*)` | no (counts rows) | fastest row count |
| `COUNT(col)` | yes | counts non-NULL values |
| `COUNT(DISTINCT col)` | yes | distinct non-NULL values |
| `SUM/AVG/MIN/MAX` | yes | `AVG` skips NULLs — mind denominators |
| `ROUND(x, n)` | n/a | rounding, not aggregation |
| `GROUP BY` | — | every selected non-agg column must appear here |
| `HAVING` | — | aggregated filter; `WHERE` cannot use aggregates |

> **Gotcha:** `WHERE salary > AVG(salary)` is illegal (aggregate on a row level).
> Use a subquery or a window function: `AVG(salary) OVER ()` (§9).

---
## 6. String & date functions (SQLite flavour)

▶ Session on Datetime in SQL: https://www.youtube.com/watch?v=OY9YiALhSR4
▶ SQL Datetime Case Study on Flights: https://www.youtube.com/watch?v=zY1UCTko64E

**NOTE (dialect):** names shift across engines — MySQL `CONCAT()/DATE_FORMAT()`,
PostgreSQL `||`/`EXTRACT()/to_char()`, SQL Server `LEN()/GETDATE()`. The
*concepts* are identical.

In [25]:
# strings: UPPER LOWER LENGTH SUBSTR TRIM REPLACE and || (concat)
print(q("""
    SELECT emp_name,
           UPPER(emp_name)           AS upper_name,
           LOWER(emp_name)           AS lower_name,
           LENGTH(emp_name)          AS name_len,
           SUBSTR(emp_name, 1, 3)    AS first_3,
           TRIM('   padded   ')      AS trimmed,
           REPLACE(city, 'Bengaluru', 'BLR') AS city_short,
           'Mr. ' || emp_name        AS salutation   -- || is string concat
    FROM employees
    LIMIT 5
"""))
print("\nNULL concat is NULL (SQL 3-valued logic):", q("SELECT 'a' || NULL AS x"))


  emp_name upper_name lower_name  name_len first_3 trimmed city_short  \
0    Rahul      RAHUL      rahul         5     Rah  padded        BLR   
1    Priya      PRIYA      priya         5     Pri  padded        BLR   
2     Amit       AMIT       amit         4     Ami  padded     Mumbai   
3    Sneha      SNEHA      sneha         5     Sne  padded     Mumbai   
4   Vikram     VIKRAM     vikram         6     Vik  padded      Delhi   

   salutation  
0   Mr. Rahul  
1   Mr. Priya  
2    Mr. Amit  
3   Mr. Sneha  
4  Mr. Vikram  

NULL concat is NULL (SQL 3-valued logic):       x
0  None


In [26]:
# numbers: ROUND, integer division gotcha, CAST
print(q("""
    SELECT ROUND(3.14159, 2) AS rounded,
           7 / 2             AS int_div,      -- 3  (both ints -> integer division in most engines)
           7.0 / 2           AS real_div,     -- 3.5
           CAST('42' AS INT) + 1 AS casted,
           ABS(-45)          AS absolute
"""))
# NOTE: SQLite's / between two INTEGERs IS integer division — cast one side to REAL.


   rounded  int_div  real_div  casted  absolute
0     3.14        3       3.5      43        45


In [27]:
# dates: strftime is the workhorse
print(q("""
    SELECT hire_date,
           strftime('%Y', hire_date)  AS yr,
           strftime('%m', hire_date)  AS mo,
           strftime('%d', hire_date)  AS dy,
           strftime('%Y-%m', hire_date) AS ym,
           strftime('%w', hire_date)  AS weekday_num,   -- 0 = Sunday
           date(hire_date, '+30 days')  AS plus_30d,
           date(hire_date, '-1 year')   AS minus_1y
    FROM employees
    ORDER BY hire_date
    LIMIT 6
"""))


    hire_date    yr  mo  dy       ym weekday_num    plus_30d    minus_1y
0  2016-05-05  2016  05  05  2016-05           4  2016-06-04  2015-05-05
1  2017-11-20  2017  11  20  2017-11           1  2017-12-20  2016-11-20
2  2018-02-18  2018  02  18  2018-02           0  2018-03-20  2017-02-18
3  2019-03-15  2019  03  15  2019-03           5  2019-04-14  2018-03-15
4  2019-12-30  2019  12  30  2019-12           1  2020-01-29  2018-12-30
5  2020-02-14  2020  02  14  2020-02           5  2020-03-15  2019-02-14


In [28]:
# date arithmetic, "now", and EXTRACT equivalents
print(q("""
    SELECT date('now')                    AS today,
           datetime('now', '-2 days')     AS two_days_ago,
           date('2024-01-15', '+1 month') AS plus_a_month,
           julianday('2024-03-10') - julianday('2024-03-01') AS day_gap,
           CAST(strftime('%Y', '2024-03-15') AS INT) AS extract_year
"""))
# NOTE (dialect): other engines use EXTRACT(YEAR FROM d), DATEADD(month, 1, d),
# DATEDIFF(...), NOW()/CURRENT_DATE. SQLite: strftime + date/time modifiers.


        today         two_days_ago plus_a_month  day_gap  extract_year
0  2026-10-07  2026-10-05 18:08:49   2024-02-15      9.0          2024


In [29]:
# NULL helpers: COALESCE (first non-null), IFNULL (2-arg SQLite), NULLIF (equal -> NULL)
print(q("""
    SELECT emp_name, bonus,
           COALESCE(bonus, 0)                    AS bonus_or_zero,
           IFNULL(city, 'unknown')               AS city_safe,
           NULLIF(dept_id, 6)                    AS null_if_support,   -- 6 -> NULL
           COALESCE(bonus, 0) * 2                AS bonus_double
    FROM employees
    ORDER BY emp_id
    LIMIT 6
"""))


  emp_name   bonus  bonus_or_zero  city_safe  null_if_support  bonus_double
0    Rahul  5000.0         5000.0  Bengaluru                2       10000.0
1    Priya     NaN            0.0  Bengaluru                2           0.0
2     Amit  3000.0         3000.0     Mumbai                1        6000.0
3    Sneha     NaN            0.0     Mumbai                1           0.0
4   Vikram  8000.0         8000.0      Delhi                3       16000.0
5     Neha     NaN            0.0      Delhi                3           0.0


**Section notes — functions:**

| Need | SQLite | MySQL | PostgreSQL |
|---|---|---|---|
| concat | `\|\|` | `CONCAT()` | `\|\|` |
| upper/lower/len | `UPPER/LOWER/LENGTH` | same (`CHAR_LENGTH`) | `UPPER/LOWER/LENGTH` |
| substring | `SUBSTR(s, from, len)` | `SUBSTR`/`SUBSTRING` | `SUBSTRING` |
| trim | `TRIM(s)` | `TRIM/LTRIM/RTRIM` | `TRIM` |
| replace | `REPLACE(s, a, b)` | same | `REPLACE` |
| round | `ROUND(x, n)` | same | same |
| date parts | `strftime('%Y', d)` | `YEAR(d)`, `DATE_FORMAT` | `EXTRACT(YEAR FROM d)` |
| now | `datetime('now')` | `NOW()` | `NOW()` |
| day diff | `julianday(a) - julianday(b)` | `DATEDIFF(a, b)` | `a - b` (date) |
| nulls | `COALESCE/IFNULL/NULLIF` | `IFNULL` exists | `COALESCE/NULLIF` |

---
## 7. Joins

▶ Session 34 - SQL Joins: https://www.youtube.com/watch?v=s6xAVqYdmJ4

| Join | Rows returned | "venn" |
|---|---|---|
| `INNER` | matching rows only | A ∩ B |
| `LEFT` | all of A + matches from B (NULL if none) | A ⊇ (A ∩ B) |
| `RIGHT` | all of B + matches from A | B ⊇ (A ∩ B) — *swap the sides on SQLite < 3.39* |
| `FULL OUTER` | all of A and all of B | A ∪ B — *emulated below* |
| `CROSS` | every pair (n × m rows) | A × B |
| self | a table joined to **itself** | hierarchy/manager lookup |

In [30]:
# INNER JOIN — only matches
print(q("""
    SELECT e.emp_name, d.dept_name, d.location
    FROM employees e
    INNER JOIN departments d ON e.dept_id = d.dept_id
    ORDER BY d.dept_name, e.emp_name
"""))


   emp_name  dept_name   location
0      Neha    Finance      Delhi
1    Vikram    Finance      Delhi
2      Amit         HR     Mumbai
3     Sneha         HR     Mumbai
4     Priya         IT  Bengaluru
5     Rahul         IT  Bengaluru
6     Rohan         IT  Bengaluru
7     Divya  Marketing     Mumbai
8     Arjun      Sales       Pune
9     Kavya      Sales       Pune
10   Manish    Support  Hyderabad
11    Pooja    Support  Hyderabad


In [31]:
# LEFT JOIN — keep every employee even if dept were missing (it isn't, but Legal...)
print(q("""
    SELECT d.dept_name, COUNT(e.emp_id) AS headcount
    FROM departments d
    LEFT JOIN employees e ON e.dept_id = d.dept_id
    GROUP BY d.dept_name
    ORDER BY headcount DESC
"""))
# 'Legal' shows 0 — proof the LEFT side is fully preserved.


   dept_name  headcount
0         IT          3
1    Finance          2
2         HR          2
3      Sales          2
4    Support          2
5  Marketing          1
6      Legal          0


In [32]:
# RIGHT JOIN: this SQLite build (3.35.5) predates 3.39 — A RIGHT JOIN B
# is *definitionally* the same as B LEFT JOIN A, so just swap the sides:
print("emulated RIGHT JOIN (employees RIGHT JOIN departments == departments LEFT JOIN employees):\n", q("""
    SELECT e.emp_name, d.dept_name
    FROM departments d
    LEFT JOIN employees e ON e.dept_id = d.dept_id
    ORDER BY d.dept_name, e.emp_name
"""))


emulated RIGHT JOIN (employees RIGHT JOIN departments == departments LEFT JOIN employees):
    emp_name  dept_name
0      Neha    Finance
1    Vikram    Finance
2      Amit         HR
3     Sneha         HR
4     Priya         IT
5     Rahul         IT
6     Rohan         IT
7      None      Legal
8     Divya  Marketing
9     Arjun      Sales
10    Kavya      Sales
11   Manish    Support
12    Pooja    Support


In [33]:
# FULL OUTER JOIN emulation: LEFT JOIN  UNION ALL  anti-join of the right side
print("emulated FULL OUTER JOIN:\n", q("""
    SELECT e.emp_name, d.dept_name, 'matched/employee-side' AS side
    FROM employees e
    LEFT JOIN departments d ON e.dept_id = d.dept_id
    UNION ALL
    SELECT e.emp_name, d.dept_name, 'department-only' AS side
    FROM departments d
    LEFT JOIN employees e ON e.dept_id = d.dept_id
    WHERE e.emp_id IS NULL
    ORDER BY side, d.dept_name
"""))


emulated FULL OUTER JOIN:
    emp_name  dept_name                   side
0      None      Legal        department-only
1    Vikram    Finance  matched/employee-side
2      Neha    Finance  matched/employee-side
3      Amit         HR  matched/employee-side
4     Sneha         HR  matched/employee-side
5     Rahul         IT  matched/employee-side
6     Priya         IT  matched/employee-side
7     Rohan         IT  matched/employee-side
8     Divya  Marketing  matched/employee-side
9     Arjun      Sales  matched/employee-side
10    Kavya      Sales  matched/employee-side
11   Manish    Support  matched/employee-side
12    Pooja    Support  matched/employee-side


In [34]:
# CROSS JOIN (7 depts × 12 employees = 84 pairs) and a non-equi join
print("cross join row count:", q("""
    SELECT COUNT(*) AS n FROM departments CROSS JOIN employees
""")["n"][0])

con.executescript("""
CREATE TABLE salary_bands(band TEXT, lo REAL, hi REAL);
INSERT INTO salary_bands VALUES
 ('Junior',      0, 60000),
 ('Mid',     60000, 85000),
 ('Senior',  85000, 1000000);
""")

print("\nnon-equi join — match employees to a RANGE table (condition is not '='):\n", q("""
    SELECT e.emp_name, e.salary, b.band
    FROM employees e
    JOIN salary_bands b
      ON e.salary >= b.lo AND e.salary < b.hi      -- inequality, no equality involved
    ORDER BY e.salary DESC
"""))


cross join row count: 84

non-equi join — match employees to a RANGE table (condition is not '='):
    emp_name   salary    band
0     Rahul  90000.0  Senior
1     Rohan  90000.0  Senior
2    Vikram  88000.0  Senior
3     Priya  75000.0     Mid
4      Neha  70000.0     Mid
5     Arjun  66000.0     Mid
6      Amit  62000.0     Mid
7     Divya  60000.0     Mid
8     Sneha  58000.0  Junior
9     Kavya  54000.0  Junior
10   Manish  48000.0  Junior
11    Pooja  46000.0  Junior


In [35]:
# SELF JOIN — employees and THEIR managers
print(q("""
    SELECT e.emp_name            AS employee,
           COALESCE(m.emp_name, '(no manager)') AS manager,
           e.dept_id
    FROM employees e
    LEFT JOIN employees m ON e.manager_id = m.emp_id
    ORDER BY manager, employee
"""))


   employee       manager  dept_id
0      Amit  (no manager)        1
1     Arjun  (no manager)        4
2     Divya  (no manager)        5
3    Manish  (no manager)        6
4     Rahul  (no manager)        2
5    Vikram  (no manager)        3
6     Sneha          Amit        1
7     Kavya         Arjun        4
8     Pooja        Manish        6
9     Priya         Rahul        2
10    Rohan         Rahul        2
11     Neha        Vikram        3


In [36]:
# join on MULTIPLE keys + join ... WHERE gotcha (filter in ON vs WHERE)
print("join on two keys:\n", q("""
    SELECT o.order_id, c.cust_name, c.city, o.amount
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id AND c.city = 'Delhi'
    ORDER BY o.order_id
"""))

left = "SELECT COUNT(*) AS n FROM departments d LEFT JOIN employees e ON e.dept_id = d.dept_id"
with_filter = "SELECT COUNT(*) AS n FROM departments d LEFT JOIN employees e ON e.dept_id = d.dept_id WHERE e.emp_id IS NOT NULL"
print("\nLEFT JOIN rows                :", q(left)["n"][0])
print("LEFT JOIN + WHERE e.emp_id... :", q(with_filter)["n"][0],
      " <- WHERE on the right table silently turns LEFT into INNER!")


join on two keys:
    order_id cust_name   city  amount
0         1     Amita  Delhi  1200.0
1         2     Amita  Delhi   800.0
2         3     Amita  Delhi  1500.0
3         9    Rakesh  Delhi  1800.0

LEFT JOIN rows                : 13
LEFT JOIN + WHERE e.emp_id... : 12  <- WHERE on the right table silently turns LEFT into INNER!


> **Gotcha:** put filters on the **right** table inside the `ON` clause to
> keep the left table intact; filters in `WHERE` run *after* the join and
> cancel a LEFT JOIN for any row that failed the match.

**Section notes — joins:**

| Goal | Query |
|---|---|
| matches only | `FROM a JOIN b ON a.k = b.k` |
| keep all of a | `FROM a LEFT JOIN b ON ...` |
| keep all of b | `FROM b LEFT JOIN a ON ...` (aka RIGHT JOIN) |
| keep everything | `LEFT JOIN ... UNION ALL ... anti-join` (SQLite < 3.39) |
| all pairs | `FROM a CROSS JOIN b` |
| hierarchy | `FROM t x LEFT JOIN t y ON x.manager_id = y.id` |

---
## 8. Subqueries & CTEs

▶ Session 35 - Subqueries in SQL: https://www.youtube.com/watch?v=YYq47MN3TZI

In [37]:
# 1. SCALAR subquery — a single value usable like a constant
print(q("""
    SELECT emp_name, salary,
           (SELECT MAX(salary) FROM employees)          AS company_max,
           ROUND(100.0 * salary / (SELECT MAX(salary) FROM employees), 1) AS pct_of_max
    FROM employees
    ORDER BY salary DESC
    LIMIT 5
"""))


  emp_name   salary  company_max  pct_of_max
0    Rahul  90000.0      90000.0       100.0
1    Rohan  90000.0      90000.0       100.0
2   Vikram  88000.0      90000.0        97.8
3    Priya  75000.0      90000.0        83.3
4     Neha  70000.0      90000.0        77.8


In [38]:
# 2. IN / NOT IN subquery — customers who never ordered
print("customers with NO orders (NOT IN):\n", q("""
    SELECT cust_name FROM customers
    WHERE customer_id NOT IN (SELECT customer_id FROM orders)
    ORDER BY cust_name
"""))
print("customers WITH orders (IN):\n", q("""
    SELECT cust_name FROM customers
    WHERE customer_id IN (SELECT DISTINCT customer_id FROM orders)
    ORDER BY cust_name
"""))
# NOTE: NOT IN returns NOTHING if the subquery yields any NULL — prefer NOT EXISTS.


customers with NO orders (NOT IN):
   cust_name
0     Imran
1      Tara
customers WITH orders (IN):
   cust_name
0     Amita
1   Karthik
2     Meera
3     Nisha
4    Rakesh
5     Sunil
6     Vivek
7      Zoya


In [39]:
# 3. EXISTS / NOT EXISTS — correlated, short-circuits per row
print("NOT EXISTS version (same answer, safer):\n", q("""
    SELECT cust_name FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
"""))
print("\nEXISTS — departments that have at least one employee:\n", q("""
    SELECT dept_name FROM departments d
    WHERE EXISTS (SELECT 1 FROM employees e WHERE e.dept_id = d.dept_id)
"""))


NOT EXISTS version (same answer, safer):
   cust_name
0      Tara
1     Imran

EXISTS — departments that have at least one employee:
    dept_name
0         HR
1         IT
2    Finance
3      Sales
4  Marketing
5    Support


In [40]:
# 4. CORRELATED subquery — the inner query re-runs ONCE PER OUTER ROW
print("employees earning more than their OWN department average:\n", q("""
    SELECT e.emp_name, e.dept_id, e.salary
    FROM employees e
    WHERE e.salary > (
        SELECT AVG(salary) FROM employees sub WHERE sub.dept_id = e.dept_id
    )
    ORDER BY e.dept_id, e.salary DESC
"""))
# "correlated" = inner query references e.dept_id → cannot run alone;
# think of it as a per-row lookup table.


employees earning more than their OWN department average:
   emp_name  dept_id   salary
0     Amit        1  62000.0
1    Rahul        2  90000.0
2    Rohan        2  90000.0
3   Vikram        3  88000.0
4    Arjun        4  66000.0
5   Manish        6  48000.0


In [41]:
# 5. DERIVED TABLE — subquery in FROM (must be aliased)
print(q("""
    SELECT g.dept_name, g.headcount, g.avg_sal
    FROM (
        SELECT d.dept_name,
               COUNT(e.emp_id)   AS headcount,
               ROUND(AVG(e.salary)) AS avg_sal
        FROM employees e
        JOIN departments d ON e.dept_id = d.dept_id
        GROUP BY d.dept_name
    ) AS g                      -- <- alias is mandatory
    WHERE g.headcount >= 2
    ORDER BY g.avg_sal DESC
"""))


  dept_name  headcount  avg_sal
0        IT          3  85000.0
1   Finance          2  79000.0
2        HR          2  60000.0
3     Sales          2  60000.0
4   Support          2  47000.0


In [42]:
# 6. CTE with WITH — named, readable subquery (may be referenced multiple times)
print(q("""
    WITH dept_stats AS (
        SELECT dept_id, COUNT(*) AS n, AVG(salary) AS avg_sal
        FROM employees
        GROUP BY dept_id
    ),
    ranked AS (
        SELECT dept_id, n, ROUND(avg_sal) AS avg_sal,
               RANK() OVER (ORDER BY avg_sal DESC) AS rnk
        FROM dept_stats
    )
    SELECT r.dept_id, d.dept_name, r.n, r.avg_sal, r.rnk
    FROM ranked r
    JOIN departments d ON d.dept_id = r.dept_id
    ORDER BY r.rnk
"""))


   dept_id  dept_name  n  avg_sal  rnk
0        2         IT  3  85000.0    1
1        3    Finance  2  79000.0    2
2        1         HR  2  60000.0    3
3        4      Sales  2  60000.0    3
4        5  Marketing  1  60000.0    3
5        6    Support  2  47000.0    6


In [43]:
# 6b. RECURSIVE CTE — org hierarchy (manager → reports → their reports ...)
con.executescript("""
CREATE TABLE org_tree(
    emp_id INTEGER PRIMARY KEY,
    emp_name TEXT,
    manager_id INTEGER
);
INSERT INTO org_tree VALUES
 (1, 'CEO Priya',  NULL),
 (2, 'Rahul',      1),
 (3, 'Vikram',     1),
 (4, 'Priya Eng',  2),
 (5, 'Rohan',      2),
 (6, 'Neha',       3),
 (7, 'Pooja',      6);
""")

print(q("""
    WITH RECURSIVE chain(emp_id, emp_name, manager_id, depth, path) AS (
        SELECT emp_id, emp_name, manager_id, 0, emp_name       -- anchor member
        FROM org_tree WHERE manager_id IS NULL
        UNION ALL
        SELECT c.emp_id, c.emp_name, c.manager_id, ch.depth + 1,
               ch.path || ' > ' || c.emp_name                   -- recursive member
        FROM org_tree c
        JOIN chain ch ON c.manager_id = ch.emp_id
    )
    SELECT printf('%*s%s', depth * 4, '', emp_name) AS org,
           depth, path
    FROM chain ORDER BY path
"""))
# anchor (root) runs once; the recursive part joins back until no new rows appear.


                 org  depth                               path
0          CEO Priya      0                          CEO Priya
1              Rahul      1                  CEO Priya > Rahul
2          Priya Eng      2      CEO Priya > Rahul > Priya Eng
3              Rohan      2          CEO Priya > Rahul > Rohan
4             Vikram      1                 CEO Priya > Vikram
5               Neha      2          CEO Priya > Vikram > Neha
6              Pooja      3  CEO Priya > Vikram > Neha > Pooja


In [44]:
# 7. ANY / ALL — NOT supported by SQLite; rewrite with aggregates instead:
#    WHERE salary > ANY (SELECT ...)   ==  WHERE salary > (SELECT MIN(...) ...)
#    WHERE salary > ALL (SELECT ...)   ==  WHERE salary > (SELECT MAX(...) ...)
print("ALL rewritten with MAX():\n", q("""
    SELECT emp_name, salary FROM employees
    WHERE salary > (SELECT MAX(salary) FROM employees WHERE dept_id = 1)
"""))
print("\nANY rewritten with MIN():\n", q("""
    SELECT dept_id, emp_name, salary FROM employees e
    WHERE salary > (SELECT MIN(salary) FROM employees WHERE dept_id = e.dept_id)
    ORDER BY dept_id, salary DESC
"""))


ALL rewritten with MAX():
   emp_name   salary
0    Rahul  90000.0
1    Priya  75000.0
2   Vikram  88000.0
3     Neha  70000.0
4    Arjun  66000.0
5    Rohan  90000.0

ANY rewritten with MIN():
    dept_id emp_name   salary
0        1     Amit  62000.0
1        2    Rahul  90000.0
2        2    Rohan  90000.0
3        3   Vikram  88000.0
4        4    Arjun  66000.0
5        6   Manish  48000.0


**Section notes — subqueries:**

| Type | Where it sits | Runs |
|---|---|---|
| scalar | any expression position | once |
| `IN (subquery)` | `WHERE` | once (semi-join) |
| `EXISTS` | `WHERE` | short-circuits per row |
| correlated | `WHERE` with outer ref | **once per outer row** |
| derived | `FROM` (aliased) | once |
| CTE `WITH` | before `SELECT` | named, reusable (may be referenced twice) |
| recursive CTE | `WITH RECURSIVE` | anchor + loop until empty |
| `ANY/ALL` | `WHERE` | ❌ SQLite → use `MIN()`/`MAX()` |

---
## 9. Window functions

▶ Session 36 - Window Functions in SQL: https://www.youtube.com/watch?v=3LJ-yWBQBGs
▶ Session 37 - Window Functions Part 2: https://www.youtube.com/watch?v=YgizUxmBFK8
▶ Session 37 (Cont..) - Window Functions Part 3: https://www.youtube.com/watch?v=e0HQVOdfqlQ

A window function computes across a **set of rows related to the current row**
**without collapsing them** — that is the whole difference from `GROUP BY`.

In [45]:
# ### ROW_NUMBER vs RANK vs DENSE_RANK (ties break them apart)
print(q("""
    SELECT emp_name, dept_id, salary,
           ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS rn,
           RANK()       OVER (PARTITION BY dept_id ORDER BY salary DESC) AS rnk,
           DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS dense_rnk
    FROM employees
    WHERE dept_id = 2            -- IT: Rahul & Rohan TIE at 90000
    ORDER BY salary DESC
"""))
# ROW_NUMBER: always 1,2,3 (ties broken arbitrarily)
# RANK:       1,1,3  (gap after ties)
# DENSE_RANK: 1,1,2  (no gap)


  emp_name  dept_id   salary  rn  rnk  dense_rnk
0    Rahul        2  90000.0   1    1          1
1    Rohan        2  90000.0   2    1          1
2    Priya        2  75000.0   3    3          2


In [46]:
# NTILE — split rows into n balanced buckets
print(q("""
    SELECT emp_name, salary,
           NTILE(4) OVER (ORDER BY salary DESC) AS quartile
    FROM employees
    ORDER BY salary DESC
"""))


   emp_name   salary  quartile
0     Rahul  90000.0         1
1     Rohan  90000.0         1
2    Vikram  88000.0         1
3     Priya  75000.0         2
4      Neha  70000.0         2
5     Arjun  66000.0         2
6      Amit  62000.0         3
7     Divya  60000.0         3
8     Sneha  58000.0         3
9     Kavya  54000.0         4
10   Manish  48000.0         4
11    Pooja  46000.0         4


In [47]:
# LAG / LEAD — look at previous / next row (with optional default)
print(q("""
    SELECT emp_name, hire_date, salary,
           LAG(salary)  OVER (ORDER BY hire_date)        AS prev_hire_salary,
           LEAD(salary) OVER (ORDER BY hire_date)        AS next_hire_salary,
           LAG(salary, 1, 0) OVER (ORDER BY hire_date)   AS prev_or_zero
    FROM employees
    ORDER BY hire_date
"""))
# LAG(col, n, default): n = rows back (1 by default), default shown when no row exists.


   emp_name   hire_date   salary  prev_hire_salary  next_hire_salary  \
0    Vikram  2016-05-05  88000.0               NaN           62000.0   
1      Amit  2017-11-20  62000.0           88000.0           66000.0   
2     Arjun  2018-02-18  66000.0           62000.0           90000.0   
3     Rahul  2019-03-15  90000.0           66000.0           60000.0   
4     Divya  2019-12-30  60000.0           90000.0           48000.0   
5    Manish  2020-02-14  48000.0           60000.0           75000.0   
6     Priya  2020-07-01  75000.0           48000.0           58000.0   
7     Sneha  2021-01-10  58000.0           75000.0           90000.0   
8     Rohan  2021-06-25  90000.0           58000.0           70000.0   
9      Neha  2022-09-12  70000.0           90000.0           54000.0   
10    Kavya  2023-04-01  54000.0           70000.0           46000.0   
11    Pooja  2024-01-08  46000.0           54000.0               NaN   

    prev_or_zero  
0            0.0  
1        88000.0  
2     

In [48]:
# FIRST_VALUE / LAST_VALUE — head/tail of the window (frame matters!)
print(q("""
    SELECT emp_name, dept_id, hire_date,
           FIRST_VALUE(emp_name) OVER (
               PARTITION BY dept_id ORDER BY hire_date
               ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
           ) AS first_hired_in_dept,
           LAST_VALUE(emp_name) OVER (
               PARTITION BY dept_id ORDER BY hire_date
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ) AS most_recent_upto_now
    FROM employees
    ORDER BY dept_id, hire_date
"""))
# WARNING: LAST_VALUE's default frame ends at CURRENT ROW — without an explicit
# frame you silently get the current row, not the last row of the group.


   emp_name  dept_id   hire_date first_hired_in_dept most_recent_upto_now
0      Amit        1  2017-11-20                Amit                 Amit
1     Sneha        1  2021-01-10                Amit                Sneha
2     Rahul        2  2019-03-15               Rahul                Rahul
3     Priya        2  2020-07-01               Rahul                Priya
4     Rohan        2  2021-06-25               Rahul                Rohan
5    Vikram        3  2016-05-05              Vikram               Vikram
6      Neha        3  2022-09-12              Vikram                 Neha
7     Arjun        4  2018-02-18               Arjun                Arjun
8     Kavya        4  2023-04-01               Arjun                Kavya
9     Divya        5  2019-12-30               Divya                Divya
10   Manish        6  2020-02-14              Manish               Manish
11    Pooja        6  2024-01-08              Manish                Pooja


In [49]:
# RUNNING TOTAL — SUM() OVER (ORDER BY ... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
print(q("""
    SELECT order_id, order_date, amount,
           SUM(amount) OVER (
               ORDER BY order_date, order_id
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ) AS running_total
    FROM orders
    ORDER BY order_date, order_id
"""))


    order_id  order_date  amount  running_total
0          1  2024-01-05  1200.0         1200.0
1          2  2024-01-06   800.0         2000.0
2          4  2024-01-06  2200.0         4200.0
3          3  2024-01-20  1500.0         5700.0
4          5  2024-02-03   900.0         6600.0
5          6  2024-02-05  3000.0         9600.0
6          7  2024-02-06   450.0        10050.0
7          8  2024-02-07   700.0        10750.0
8          9  2024-02-14  1800.0        12550.0
9         10  2024-02-14   600.0        13150.0
10        11  2024-03-02  2500.0        15650.0
11        12  2024-03-08  1100.0        16750.0
12        13  2024-03-15   950.0        17700.0
13        14  2024-03-16  1300.0        19000.0
14        15  2024-03-21  2000.0        21000.0


In [50]:
# MOVING AVERAGE (3-row window: current + 2 preceding)
print(q("""
    SELECT order_id, amount,
           ROUND(AVG(amount) OVER (
               ORDER BY order_id
               ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
           ), 1) AS moving_avg_3
    FROM orders
    ORDER BY order_id
"""))


    order_id  amount  moving_avg_3
0          1  1200.0        1200.0
1          2   800.0        1000.0
2          3  1500.0        1166.7
3          4  2200.0        1500.0
4          5   900.0        1533.3
5          6  3000.0        2033.3
6          7   450.0        1450.0
7          8   700.0        1383.3
8          9  1800.0         983.3
9         10   600.0        1033.3
10        11  2500.0        1633.3
11        12  1100.0        1400.0
12        13   950.0        1516.7
13        14  1300.0        1116.7
14        15  2000.0        1416.7


In [51]:
# RANKING TOP-N PER GROUP — ROW_NUMBER (exactly N) vs RANK (include ties)
print("top 2 earners per department (ROW_NUMBER — exactly 2 rows each):\n", q("""
    SELECT * FROM (
        SELECT emp_name, dept_id, salary,
               ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn
        FROM employees
    )
    WHERE rn <= 2
    ORDER BY dept_id, rn
"""))

print("\nrank of every IT employee (RANK keeps the tie visible):\n", q("""
    SELECT emp_name, salary,
           RANK()       OVER (ORDER BY salary DESC) AS rnk,
           ROW_NUMBER() OVER (ORDER BY salary DESC) AS rn
    FROM employees
    WHERE dept_id = 2
"""))


top 2 earners per department (ROW_NUMBER — exactly 2 rows each):
    emp_name  dept_id   salary  rn
0      Amit        1  62000.0   1
1     Sneha        1  58000.0   2
2     Rahul        2  90000.0   1
3     Rohan        2  90000.0   2
4    Vikram        3  88000.0   1
5      Neha        3  70000.0   2
6     Arjun        4  66000.0   1
7     Kavya        4  54000.0   2
8     Divya        5  60000.0   1
9    Manish        6  48000.0   1
10    Pooja        6  46000.0   2

rank of every IT employee (RANK keeps the tie visible):
   emp_name   salary  rnk  rn
0    Rahul  90000.0    1   1
1    Rohan  90000.0    1   2
2    Priya  75000.0    3   3


In [52]:
# AGGREGATE WINDOW vs GROUP BY — same number, rows preserved
print("window: every row kept, group stats attached:\n", q("""
    SELECT emp_name, dept_id, salary,
           COUNT(*)    OVER (PARTITION BY dept_id)          AS dept_size,
           ROUND(AVG(salary) OVER (PARTITION BY dept_id))   AS dept_avg,
           salary - ROUND(AVG(salary) OVER (PARTITION BY dept_id)) AS vs_avg
    FROM employees
    ORDER BY dept_id, salary DESC
"""))

print("\nGROUP BY: one row per group (detail is gone):\n", q("""
    SELECT dept_id, COUNT(*) AS dept_size, ROUND(AVG(salary)) AS dept_avg
    FROM employees
    GROUP BY dept_id
"""))


window: every row kept, group stats attached:
    emp_name  dept_id   salary  dept_size  dept_avg   vs_avg
0      Amit        1  62000.0          2   60000.0   2000.0
1     Sneha        1  58000.0          2   60000.0  -2000.0
2     Rahul        2  90000.0          3   85000.0   5000.0
3     Rohan        2  90000.0          3   85000.0   5000.0
4     Priya        2  75000.0          3   85000.0 -10000.0
5    Vikram        3  88000.0          2   79000.0   9000.0
6      Neha        3  70000.0          2   79000.0  -9000.0
7     Arjun        4  66000.0          2   60000.0   6000.0
8     Kavya        4  54000.0          2   60000.0  -6000.0
9     Divya        5  60000.0          1   60000.0      0.0
10   Manish        6  48000.0          2   47000.0   1000.0
11    Pooja        6  46000.0          2   47000.0  -1000.0

GROUP BY: one row per group (detail is gone):
    dept_id  dept_size  dept_avg
0        1          2   60000.0
1        2          3   85000.0
2        3          2   79000

### `OVER()` anatomy

```text
function(args) OVER (
    PARTITION BY col        -- split into groups (like GROUP BY, per row)
    ORDER BY col [ASC|DESC] -- order inside each group
    ROWS BETWEEN a AND b    -- physical frame (rows relative to current row)
)
```

| Frame | Meaning |
|---|---|
| `UNBOUNDED PRECEDING → CURRENT ROW` | start of partition → now (running) |
| `2 PRECEDING → CURRENT ROW` | 3-row moving window |
| `UNBOUNDED PRECEDING → UNBOUNDED FOLLOWING` | whole partition |
| `BETWEEN CURRENT ROW AND 1 FOLLOWING` | now → next row |

| | `GROUP BY` | window `OVER` |
|---|---|---|
| output rows | 1 per group | 1 per **input** row |
| detail columns | must be grouped/aggregated | fully preserved |
| can mix group + row values | ❌ | ✅ (`salary - AVG(...) OVER (...)`) |
| typical use | reports, summaries | rankings, running totals, MoM |

**Section notes — windows:**

| Need | Function |
|---|---|
| sequential id in group | `ROW_NUMBER() OVER (PARTITION BY g ORDER BY x)` |
| dense competition rank | `RANK()` / `DENSE_RANK()` |
| buckets | `NTILE(n)` |
| previous/next value | `LAG(col, n, default)` / `LEAD(...)` |
| first/last in group | `FIRST_VALUE` / `LAST_VALUE` (+ explicit frame!) |
| running total | `SUM() OVER (ORDER BY d ROWS UNBOUNDED PRECEDING→CURRENT)` |
| moving average | `AVG() OVER (ORDER BY d ROWS 2 PRECEDING→CURRENT)` |
| per-group stats | aggregate + `OVER (PARTITION BY g)` |

---
## 10. Set operations

▶ Session 34 - SQL Joins: https://www.youtube.com/watch?v=s6xAVqYdmJ4

In [53]:
# UNION (removes duplicates) vs UNION ALL (keeps them — faster)
print("UNION     :", q("SELECT city FROM customers UNION SELECT city FROM employees ORDER BY city")["city"].tolist())
print("UNION ALL :", len(q("SELECT city FROM customers UNION ALL SELECT city FROM employees")), "rows (with dupes)")

dupes = q("""
    SELECT city FROM customers
    UNION ALL
    SELECT city FROM employees
""")["city"].value_counts()
print("dupes UNION ALL would remove:\n", dupes[dupes > 1])


UNION     : ['Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai', 'Pune']
UNION ALL : 22 rows (with dupes)
dupes UNION ALL would remove:
 city
Delhi        5
Mumbai       5
Pune         4
Hyderabad    3
Bengaluru    3
Name: count, dtype: int64


In [54]:
# INTERSECT (in both) and EXCEPT (in first, not second)
print("cities in BOTH customers and employees (INTERSECT):\n",
      q("SELECT city FROM customers INTERSECT SELECT city FROM employees ORDER BY city"))

print("\nemployee cities NOT in customers (EXCEPT):\n",
      q("SELECT city FROM employees EXCEPT SELECT city FROM customers ORDER BY city"))


cities in BOTH customers and employees (INTERSECT):
         city
0      Delhi
1  Hyderabad
2     Mumbai
3       Pune

employee cities NOT in customers (EXCEPT):
         city
0  Bengaluru


| Operator | Keeps dupes? | Rows |
|---|---|---|
| `UNION` | no (distinct) | distinct rows of A ∪ B |
| `UNION ALL` | yes | every row of A and B (cheaper) |
| `INTERSECT` | no | rows in **both** |
| `EXCEPT` | no | rows in A **not** in B |

> **Gotcha:** all arms must have the **same number of columns** with compatible
> types; `ORDER BY` applies to the *whole* result and goes at the very end.
> Set ops compare NULLs as *not equal* in some engines — test before relying on it.

---
## 11. Views & indexes

▶ Session on Views and UDFs in SQL: https://www.youtube.com/watch?v=dMoKTx1bVnY

In [55]:
# CREATE VIEW — a saved query, not a copy of the data
con.execute("""
    CREATE VIEW v_emp_overview AS
    SELECT e.emp_id, e.emp_name, d.dept_name, e.salary, e.hire_date
    FROM employees e
    JOIN departments d ON e.dept_id = d.dept_id
""")
print(q("SELECT * FROM v_emp_overview ORDER BY salary DESC LIMIT 5"))

# views compose: query a view like a table
print("\navg salary per dept, from the view:\n", q("""
    SELECT dept_name, COUNT(*) AS n, ROUND(AVG(salary)) AS avg_sal
    FROM v_emp_overview
    GROUP BY dept_name
    ORDER BY avg_sal DESC
"""))

con.execute("DROP VIEW v_emp_overview")
print("views left:", q("SELECT COUNT(*) AS n FROM sqlite_master WHERE type='view'")["n"][0])
# NOTE: a view stores only its SQL — change the base table and the view follows.


   emp_id emp_name dept_name   salary   hire_date
0       1    Rahul        IT  90000.0  2019-03-15
1       9    Rohan        IT  90000.0  2021-06-25
2       5   Vikram   Finance  88000.0  2016-05-05
3       2    Priya        IT  75000.0  2020-07-01
4       6     Neha   Finance  70000.0  2022-09-12



avg salary per dept, from the view:
    dept_name  n  avg_sal
0         IT  3  85000.0
1    Finance  2  79000.0
2      Sales  2  60000.0
3  Marketing  1  60000.0
4         HR  2  60000.0
5    Support  2  47000.0
views left: 0


In [56]:
# EXPLAIN QUERY PLAN — see whether an index is used
plan_before = q("EXPLAIN QUERY PLAN SELECT * FROM employees WHERE dept_id = 2")
print("BEFORE index:\n", plan_before)

con.execute("CREATE INDEX idx_emp_dept ON employees(dept_id)")
plan_after = q("EXPLAIN QUERY PLAN SELECT * FROM employees WHERE dept_id = 2")
print("\nAFTER index:\n", plan_after)
# SCAN TABLE = full table scan (O(n))  →  SEARCH ... USING INDEX (O(log n))


BEFORE index:
    id  parent  notused                detail
0   2       0        0  SCAN TABLE employees

AFTER index:
    id  parent  notused                                             detail
0   3       0        0  SEARCH TABLE employees USING INDEX idx_emp_dep...


In [57]:
# UNIQUE index — enforces uniqueness like a constraint
con.execute("CREATE UNIQUE INDEX idx_emp_name ON employees(emp_name)")
try:
    con.execute("INSERT INTO employees VALUES(99, 'Rahul', 1, NULL, 50000, NULL, '2024-01-01', 30, 'Pune')")
except sqlite3.IntegrityError as e:
    print("duplicate name rejected ->", e)

print("\nindexes on employees:",
      q("SELECT name FROM pragma_index_list('employees')")["name"].tolist())
# NOTE (dialect): PostgreSQL adds "USING gin/gin" for JSON/arrays; MySQL has
# FULLTEXT indexes. Indexes speed reads but slow writes (maintained on INSERT).


duplicate name rejected -> UNIQUE constraint failed: employees.emp_name

indexes on employees: ['idx_emp_name', 'idx_emp_dept']


**Section notes — views & indexes:**

| Object | Purpose | Cost |
|---|---|---|
| `CREATE VIEW v AS SELECT ...` | reusable query, security, simplification | none (SQL only) |
| `CREATE INDEX i ON t(col)` | fast `WHERE col = ...` lookups | slower writes, disk |
| `CREATE UNIQUE INDEX` | uniqueness without a table constraint | same |
| `EXPLAIN QUERY PLAN` | does the planner use my index? | — |
| `DROP VIEW/INDEX` | remove it | — |

> **Gotcha:** an index on `(dept_id, salary)` can serve `WHERE dept_id = ?`
> but an index on `(salary, dept_id)` **cannot** — column order matters.

---
## 12. Transactions & ACID

▶ Session on Transactions & Stored Procedures in SQL: https://www.youtube.com/watch?v=-O_9iR7XsJw

In [58]:
con.execute("CREATE TABLE accounts(acc TEXT PRIMARY KEY, bal REAL NOT NULL)")
con.execute("INSERT INTO accounts VALUES('A', 1000), ('B', 500)")
print("start:\n", q("SELECT * FROM accounts"))

# --- ROLLBACK discards the work ---
con.execute("BEGIN")
con.execute("UPDATE accounts SET bal = bal - 200 WHERE acc = 'A'")
con.execute("UPDATE accounts SET bal = bal + 200 WHERE acc = 'B'")
print("\nuncommitted (visible inside the transaction):\n", q("SELECT * FROM accounts"))
con.execute("ROLLBACK")
print("after ROLLBACK:\n", q("SELECT * FROM accounts"))

# --- COMMIT persists the work ---
con.execute("BEGIN")
con.execute("UPDATE accounts SET bal = bal - 200 WHERE acc = 'A'")
con.execute("UPDATE accounts SET bal = bal + 200 WHERE acc = 'B'")
con.execute("COMMIT")
print("\nafter COMMIT:\n", q("SELECT * FROM accounts"))

# --- SAVEPOINT = partial rollback inside a transaction ---
con.execute("BEGIN")
con.execute("UPDATE accounts SET bal = bal - 100 WHERE acc = 'A'")
con.execute("SAVEPOINT sp1")
con.execute("UPDATE accounts SET bal = bal - 100 WHERE acc = 'A'")   # this one gets undone
con.execute("ROLLBACK TO sp1")
con.execute("COMMIT")
print("\nafter SAVEPOINT rollback (A lost only the first 100):\n", q("SELECT * FROM accounts"))


start:
   acc     bal
0   A  1000.0
1   B   500.0

uncommitted (visible inside the transaction):
   acc    bal
0   A  800.0
1   B  700.0
after ROLLBACK:
   acc     bal
0   A  1000.0
1   B   500.0

after COMMIT:
   acc    bal
0   A  800.0
1   B  700.0

after SAVEPOINT rollback (A lost only the first 100):
   acc    bal
0   A  700.0
1   B  700.0


### ACID

| Property | Meaning | Example above |
|---|---|---|
| **A**tomicity | all-or-nothing: every statement in the tx commits or none does | ROLLBACK removed *both* transfers |
| **C**onsistency | tx moves the DB from one valid state to another (constraints hold) | `bal` can never violate `CHECK`/PK |
| **I**solation | concurrent tx don't see each other's uncommitted work | other sessions can't see the pending transfer |
| **D**urability | once committed, survives crashes (WAL/journal on disk) | COMMIT is permanent |

### Isolation levels & read anomalies

| Level | Dirty read | Non-repeatable read | Phantom read |
|---|---|---|---|
| READ UNCOMMITTED | ✅ possible | ✅ | ✅ |
| READ COMMITTED | prevented | ✅ | ✅ |
| REPEATABLE READ | prevented | prevented | ✅ (some engines) |
| SERIALIZABLE | prevented | prevented | prevented |

- **Dirty read** — you read another tx's *uncommitted* data.
- **Non-repeatable read** — same query twice → different value (someone committed in between).
- **Phantom read** — same query twice → different *set of rows* (inserted/deleted).
- SQLite's default: serializable (a write lock blocks other writers); readers use shared locks.

### Stored procedures — not supported by SQLite
SQLite executes statements only; there are no stored procedures/UDF blocks.
**MySQL syntax for reference** (what you would write on a server):
```sql
-- DELIMITER $$
-- CREATE PROCEDURE raise_salary(IN p_dept INT, IN pct DOUBLE)
-- BEGIN
--     UPDATE employees SET salary = salary * (1 + pct/100) WHERE dept_id = p_dept;
--     SELECT ROW_COUNT() AS updated;
-- END $$
-- DELIMITER ;
-- CALL raise_salary(2, 10);
```

**Section notes — transactions:**

| Task | Statement |
|---|---|
| start | `BEGIN` (or `BEGIN TRANSACTION`) |
| keep | `COMMIT` |
| undo all | `ROLLBACK` |
| undo part | `SAVEPOINT sp` → `ROLLBACK TO sp` |
| other dialects | `START TRANSACTION`, `COMMIT WORK`, `SET TRANSACTION ISOLATION LEVEL ...` |

---
## 13. Database design & normalization

▶ Session on Database Design | Normalization: https://www.youtube.com/watch?v=8SOs0URTvr0

### Functional dependency (FD)
`A → B` ("A functionally determines B") means each value of `A` maps to exactly
one value of `B`. Example: `emp_id → emp_name` (an id has one name) but
`city → pincode` may fail in the real world.

### The normal forms — worked example below

| Form | Rule (in one line) | Removes |
|---|---|---|
| **1NF** | every cell is atomic; no repeating groups/arrays | multi-valued cells |
| **2NF** | 1NF + no **partial** dependency (non-key col must depend on the *whole* PK) | attributes of a composite key |
| **3NF** | 2NF + no **transitive** dependency (non-key → non-key) | columns derivable from other columns |
| **BCNF** | every determinant is a candidate key | exotic FD violations (rare in practice) |

In [59]:
# UNNORMALIZED source table: repeating customer & product facts in every line
con.executescript("""
CREATE TABLE sales_unnorm (
    order_id    INTEGER,
    cust_name   TEXT,
    cust_city   TEXT,
    product     TEXT,
    qty         INTEGER,
    unit_price  REAL
);
INSERT INTO sales_unnorm VALUES
 (1, 'Amit',  'Delhi',   'Laptop',  1, 55000),
 (1, 'Amit',  'Delhi',   'Mouse',   2,   500),
 (2, 'Priya', 'Pune',    'Laptop',  1, 55000),
 (3, 'Sneha', 'Mumbai',  'Keyboard',1,  1500);
""")
print("UNNORMALIZED (1NF violations waiting to happen — redundant city/price):\n",
      q("SELECT * FROM sales_unnorm"))
# FDs here: order_id -> cust_name, cust_city   |   product -> unit_price
#           cust_city is TRANSITIVELY determined via cust_name  (3NF issue)


UNNORMALIZED (1NF violations waiting to happen — redundant city/price):
    order_id cust_name cust_city   product  qty  unit_price
0         1      Amit     Delhi    Laptop    1     55000.0
1         1      Amit     Delhi     Mouse    2       500.0
2         2     Priya      Pune    Laptop    1     55000.0
3         3     Sneha    Mumbai  Keyboard    1      1500.0


In [60]:
# normalize: customers (3NF) + products (2NF/3NF) + order_lines (1NF/2NF)
con.executescript("""
CREATE TABLE n_customers (
    cust_name TEXT PRIMARY KEY,
    cust_city TEXT NOT NULL
);
CREATE TABLE n_products (
    product    TEXT PRIMARY KEY,
    unit_price REAL NOT NULL CHECK(unit_price > 0)
);
CREATE TABLE n_order_lines (
    order_id   INTEGER,
    cust_name  TEXT REFERENCES n_customers(cust_name),
    product    TEXT REFERENCES n_products(product),
    qty        INTEGER NOT NULL CHECK(qty > 0),
    PRIMARY KEY (order_id, product)
);

INSERT INTO n_customers
    SELECT DISTINCT cust_name, cust_city FROM sales_unnorm;
INSERT INTO n_products
    SELECT DISTINCT product, unit_price FROM sales_unnorm;
INSERT INTO n_order_lines
    SELECT order_id, cust_name, product, qty FROM sales_unnorm;
""")
print("customers (3NF):\n", q("SELECT * FROM n_customers"))
print("\nproducts (3NF):\n", q("SELECT * FROM n_products"))
print("\norder_lines:\n", q("SELECT * FROM n_order_lines"))


customers (3NF):


   cust_name cust_city
0      Amit     Delhi
1     Priya      Pune
2     Sneha    Mumbai

products (3NF):
     product  unit_price
0    Laptop     55000.0
1     Mouse       500.0
2  Keyboard      1500.0

order_lines:
    order_id cust_name   product  qty
0         1      Amit    Laptop    1
1         1      Amit     Mouse    2
2         2     Priya    Laptop    1
3         3     Sneha  Keyboard    1


In [61]:
# lossless join — the normalized tables reconstruct the original exactly
rebuilt = q("""
    SELECT l.order_id, c.cust_name, c.cust_city, p.product, l.qty, p.unit_price
    FROM n_order_lines l
    JOIN n_customers c ON c.cust_name = l.cust_name
    JOIN n_products  p ON p.product   = l.product
    ORDER BY l.order_id, p.product
""")
original = q("SELECT * FROM sales_unnorm ORDER BY order_id, product")
print("rebuilt == original:", rebuilt.reset_index(drop=True).equals(original.reset_index(drop=True)))
print(rebuilt)


rebuilt == original: True
   order_id cust_name cust_city   product  qty  unit_price
0         1      Amit     Delhi    Laptop    1     55000.0
1         1      Amit     Delhi     Mouse    2       500.0
2         2     Priya      Pune    Laptop    1     55000.0
3         3     Sneha    Mumbai  Keyboard    1      1500.0


### Denormalization — the deliberate trade-off

| | Normalized (3NF) | Denormalized |
|---|---|---|
| writes | one update → one place | updates spread, redundancy risk |
| reads | joins needed (slower) | single wide table (faster) |
| anomalies | none (update/insert/delete) | can reappear |
| used for | OLTP (§1) | OLAP, data warehouses, caches |

### ER → schema rules of thumb
- 1:N → FK on the many side (our `employees.dept_id`).
- M:N → junction table with a composite PK (`order_lines` above).
- 1:1 → merge into one table, or separate if the part is large/optional.
- Prefer natural keys (`email`) when truly unique **and** stable; otherwise
  surrogate keys (`INTEGER PRIMARY KEY`).

**Section notes — design:**

| Step | Question |
|---|---|
| 1NF | is every value atomic? |
| 2NF | does each non-key column depend on the *whole* PK? |
| 3NF | does any non-key column depend on another non-key column? |
| BCNF | is every determinant a candidate key? |
| denormalize? | only when measured read cost > write/anomaly cost |

---
## 14. Data cleaning with SQL

▶ Session on Data Cleaning using SQL | Laptops: https://www.youtube.com/watch?v=NsOXJdRIamc

In [62]:
# a deliberately dirty table (raw scrape-style)
con.executescript("""
CREATE TABLE laptops_raw (
    lid    INTEGER PRIMARY KEY,
    brand  TEXT,
    model  TEXT,
    ram    TEXT,
    os     TEXT,
    rating TEXT,
    added  TEXT
);
INSERT INTO laptops_raw VALUES
 (1, ' Dell ',  'Inspiron 15', '8GB ', 'Windows 11', '4.3', '15/03/2024'),
 (2, 'dell',    'XPS 13',      ' 16 GB', 'windows11', '4.5', '2024-03-16'),
 (3, 'HP ',     'Pavilion',    '8GB',   'Windows 10', 'N/A', '2024-03-17'),
 (4, 'hp',      'EliteBook',   '16 GB', 'win10',      '4.2', '17/03/2024'),
 (5, 'Lenovo',  'IdeaPad',     '8 GB',  'Windows 11', '4.0', '2024-03-18'),
 (6, 'LENOVO',  'ThinkPad',    '32GB',  'Windows 11', '4.6', '18/03/2024'),
 (7, ' Dell ',  'Inspiron 15', '8GB ', 'Windows 11', '4.3', '15/03/2024'),
 (8, 'Asus',    'VivoBook',    NULL,    'Windows 11', NULL,  '19/03/2024');
""")
print("RAW:\n", q("SELECT * FROM laptops_raw"))


RAW:
    lid   brand        model     ram          os rating       added
0    1   Dell   Inspiron 15    8GB   Windows 11    4.3  15/03/2024
1    2    dell       XPS 13   16 GB   windows11    4.5  2024-03-16
2    3     HP      Pavilion     8GB  Windows 10    N/A  2024-03-17
3    4      hp    EliteBook   16 GB       win10    4.2  17/03/2024
4    5  Lenovo      IdeaPad    8 GB  Windows 11    4.0  2024-03-18
5    6  LENOVO     ThinkPad    32GB  Windows 11    4.6  18/03/2024
6    7   Dell   Inspiron 15    8GB   Windows 11    4.3  15/03/2024
7    8    Asus     VivoBook    None  Windows 11   None  19/03/2024


In [63]:
# 1) whitespace + case inconsistencies  ->  TRIM + UPPER + CASE
con.execute("""
    UPDATE laptops_raw SET
        brand = CASE UPPER(TRIM(brand))
                  WHEN 'DELL' THEN 'Dell'
                  WHEN 'HP'   THEN 'HP'
                  WHEN 'LENOVO' THEN 'Lenovo'
                  WHEN 'ASUS' THEN 'Asus'
                  ELSE TRIM(brand)
                END,
        os = CASE
               WHEN UPPER(os) LIKE 'WIN%' THEN 'Windows'
               ELSE os
             END
""")
print("brand/os fixed:\n", q("SELECT lid, brand, os FROM laptops_raw"))


brand/os fixed:
    lid   brand       os
0    1    Dell  Windows
1    2    Dell  Windows
2    3      HP  Windows
3    4      HP  Windows
4    5  Lenovo  Windows
5    6  Lenovo  Windows
6    7    Dell  Windows
7    8    Asus  Windows


In [64]:
# 2) recast messy numerics: ' 16 GB' -> 16 ; 'N/A' -> NULL ; rating -> REAL
con.execute("""
    UPDATE laptops_raw SET
        ram = CASE WHEN ram IS NULL THEN NULL
                   ELSE CAST(REPLACE(TRIM(ram), 'GB', '') AS INTEGER)
             END,
        rating = CASE WHEN rating = 'N/A' THEN NULL ELSE CAST(rating AS REAL) END
""")
con.execute("ALTER TABLE laptops_raw RENAME COLUMN ram TO ram_gb")
# 3) missing values -> COALESCE to a defensible default
con.execute("UPDATE laptops_raw SET ram_gb = 8 WHERE ram_gb IS NULL")          # most common
con.execute("UPDATE laptops_raw SET rating = 4.3 WHERE rating IS NULL")        # global median-ish
print("re-cast + filled:\n", q("SELECT lid, brand, ram_gb, rating FROM laptops_raw"))


re-cast + filled:
    lid   brand ram_gb rating
0    1    Dell      8    4.3
1    2    Dell     16    4.5
2    3      HP      8    4.3
3    4      HP     16    4.2
4    5  Lenovo      8    4.0
5    6  Lenovo     32    4.6
6    7    Dell      8    4.3
7    8    Asus      8    4.3


In [65]:
# 4) inconsistent dates: 'DD/MM/YYYY' and 'YYYY-MM-DD' -> ISO date
print(q("""
    SELECT lid, added,
           CASE WHEN added LIKE '__/__/____'
                THEN date(substr(added, 7, 4) || '-' || substr(added, 4, 2) || '-' || substr(added, 1, 2))
                ELSE date(added)
           END AS iso_date
    FROM laptops_raw
"""))


   lid       added    iso_date
0    1  15/03/2024  2024-03-15
1    2  2024-03-16  2024-03-16
2    3  2024-03-17  2024-03-17
3    4  17/03/2024  2024-03-17
4    5  2024-03-18  2024-03-18
5    6  18/03/2024  2024-03-18
6    7  15/03/2024  2024-03-15
7    8  19/03/2024  2024-03-19


In [66]:
# 5) deduplicate with ROW_NUMBER (keep the lowest lid per brand+model)
con.execute("""
    DELETE FROM laptops_raw
    WHERE lid IN (
        SELECT lid FROM (
            SELECT lid, ROW_NUMBER() OVER (
                       PARTITION BY brand, model ORDER BY lid
                   ) AS rn
            FROM laptops_raw
        ) WHERE rn > 1
    )
""")
print("rows after dedupe:", len(q("SELECT * FROM laptops_raw")))
print(q("SELECT lid, brand, model, ram_gb, os, rating FROM laptops_raw"))


rows after dedupe: 7
   lid   brand        model ram_gb       os rating
0    1    Dell  Inspiron 15      8  Windows    4.3
1    2    Dell       XPS 13     16  Windows    4.5
2    3      HP     Pavilion      8  Windows    4.3
3    4      HP    EliteBook     16  Windows    4.2
4    5  Lenovo      IdeaPad      8  Windows    4.0
5    6  Lenovo     ThinkPad     32  Windows    4.6
6    8    Asus     VivoBook      8  Windows    4.3


In [67]:
# 6) fix inconsistent categories with UPDATE + WHERE (targeted, safe)
con.execute("UPDATE laptops_raw SET brand = 'Asus' WHERE brand = 'asus'")
print(q("SELECT DISTINCT brand FROM laptops_raw ORDER BY brand"))


    brand
0    Asus
1    Dell
2      HP
3  Lenovo


**Section notes — SQL cleaning:**

| Problem | SQL tool |
|---|---|
| whitespace | `TRIM/LTRIM/RTRIM` |
| case | `UPPER/LOWER/CASE` |
| recast type | `CAST(col AS TYPE)` / `col * 1` |
| strip units | `REPLACE(col, 'GB', '')` then `CAST` |
| missing | `COALESCE(col, default)` / `UPDATE ... WHERE col IS NULL` |
| junk labels | `UPDATE ... CASE ...` |
| duplicates | `ROW_NUMBER() OVER (PARTITION BY keys ORDER BY id)` → `DELETE ... rn > 1` |
| bad dates | `CASE WHEN LIKE '__/__/____' THEN date(substr(...)) ELSE date(col) END` |

> **Gotcha:** always `SELECT` first, wrap `UPDATE/DELETE` in a transaction
> (`BEGIN; ...; COMMIT;`), and never mutate a raw/bronze table in place.

---
## 15. EDA with SQL

▶ Session on EDA using SQL | Laptops: https://www.youtube.com/watch?v=dv8TkUTlQvI

(Same questions as the pandas EDA checklist — answered with `GROUP BY`.)

In [68]:
# value counts
print("salary bands (value counts):\n", q("""
    SELECT CASE WHEN salary >= 85000 THEN 'Senior'
                WHEN salary >= 60000 THEN 'Mid'
                ELSE 'Junior' END AS band,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM employees), 1) AS pct
    FROM employees
    GROUP BY band
    ORDER BY n DESC
"""))

print("\norder status value counts:\n", q("""
    SELECT status, COUNT(*) AS n, SUM(amount) AS revenue
    FROM orders GROUP BY status ORDER BY n DESC
"""))


salary bands (value counts):
      band  n   pct
0     Mid  5  41.7
1  Junior  4  33.3
2  Senior  3  25.0

order status value counts:
       status   n  revenue
0  completed  12  18200.0
1  cancelled   2   2100.0
2    pending   1    700.0


In [69]:
# summary statistics — SQLite has no STDDEV()/VARIANCE(), so compute them:
#   variance = AVG(x*x) - AVG(x)*AVG(x)      (population)
#   stddev   = sqrt(variance)
print(q("""
    SELECT COUNT(*)        AS n,
           ROUND(AVG(salary), 1)   AS mean,
           ROUND(MIN(salary))      AS min,
           ROUND(MAX(salary))      AS max,
           ROUND(SUM(salary))      AS sum,
           ROUND(sqrt(AVG(salary*salary) - AVG(salary)*AVG(salary)), 1) AS stddev,
           (SELECT salary FROM employees ORDER BY salary LIMIT 1 OFFSET
               (SELECT COUNT(*)/2 FROM employees)) AS median_approx
    FROM employees
"""))
# NOTE: that sqrt formula is the POPULATION stddev (÷N). Sample stddev (÷N-1)
# needs an extra factor: sqrt(n/(n-1) * var). MySQL: STDDEV_POP/STDDEV_SAMP;
# PostgreSQL: stddev_pop/stddev_samp.


    n     mean      min      max       sum   stddev  median_approx
0  12  67250.0  46000.0  90000.0  807000.0  14995.1        66000.0


In [70]:
# correlation (Pearson) in pure SQL — same identity as in pandas .corr()
print(q("""
    WITH m AS (
        SELECT AVG(age)     AS mx,   AVG(salary) AS my,
               AVG(age*age) AS mxx,  AVG(salary*salary) AS myy,
               AVG(age*salary) AS mxy
        FROM employees
    )
    SELECT ROUND((mxy - mx*my) / (sqrt(mxx - mx*mx) * sqrt(myy - my*my)), 4) AS pearson_r_age_salary
    FROM m
"""))
# r ≈ 0 for age vs salary in this tiny sample — remember: correlation is NOT causation.


   pearson_r_age_salary
0                0.4227


In [71]:
# top-N with a tie: 3 highest salaries (RANK includes the tie -> 4 rows)
print("top 3 salaries with ties kept (RANK):\n", q("""
    SELECT emp_name, salary FROM (
        SELECT emp_name, salary, RANK() OVER (ORDER BY salary DESC) AS rnk
        FROM employees
    ) WHERE rnk <= 3
"""))
print("\ntop 3 exactly (ROW_NUMBER):\n", q("""
    SELECT emp_name, salary FROM (
        SELECT emp_name, salary, ROW_NUMBER() OVER (ORDER BY salary DESC, emp_id) AS rn
        FROM employees
    ) WHERE rn <= 3
"""))


top 3 salaries with ties kept (RANK):
   emp_name   salary
0    Rahul  90000.0
1    Rohan  90000.0
2   Vikram  88000.0

top 3 exactly (ROW_NUMBER):
   emp_name   salary
0    Rahul  90000.0
1    Rohan  90000.0
2   Vikram  88000.0


In [72]:
# time trends: monthly revenue (strftime groups the months)
print(q("""
    SELECT strftime('%Y-%m', order_date) AS month,
           COUNT(*)        AS orders,
           SUM(amount)     AS revenue,
           ROUND(AVG(amount)) AS avg_order_value
    FROM orders
    GROUP BY month
    ORDER BY month
"""))

# cohort-ish grouping: customers by signup month vs how much they ordered
print("\ncohort analysis (signup month → orders & spend):\n", q("""
    SELECT c.cohort, COUNT(DISTINCT c.customer_id) AS customers,
           COUNT(o.order_id) AS orders,
           COALESCE(SUM(o.amount), 0) AS revenue
    FROM (SELECT *, strftime('%Y-%m', signup_date) AS cohort FROM customers) c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.cohort
    ORDER BY c.cohort
"""))


     month  orders  revenue  avg_order_value
0  2024-01       4   5700.0           1425.0
1  2024-02       6   7450.0           1242.0
2  2024-03       5   7850.0           1570.0



cohort analysis (signup month → orders & spend):
     cohort  customers  orders  revenue
0  2024-01          2       5   6600.0
1  2024-02          3       6   9050.0
2  2024-03          5       4   5350.0


**Section notes — EDA in SQL:**

| Question | Query pattern |
|---|---|
| value counts | `SELECT col, COUNT(*) FROM t GROUP BY col ORDER BY 2 DESC` |
| summary stats | `MIN/MAX/AVG/SUM` + hand-rolled `sqrt(AVG(x*x)-AVG(x)*AVG(x))` |
| correlation | Pearson identity in a CTE (above) |
| top-N | `ROW_NUMBER()/RANK() OVER (...)` + outer filter |
| trends | `GROUP BY strftime('%Y-%m', date_col)` |
| cohorts | derived table of cohort label + `LEFT JOIN` + `GROUP BY cohort` |
| share of total | `100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()` (window!) |

---
## 16. SQL interview questions — 10 classics, all runnable

▶ SQL Interview Questions Part 1: https://www.youtube.com/watch?v=QlenVBvzrW4
▶ SQL Case Study 1 | Zomato Dataset (source of several of these): https://www.youtube.com/watch?v=jYwcB0KEFfY

In [73]:
# Q1 — 2nd highest salary (two idioms)
print("subquery version:\n", q("""
    SELECT MAX(salary) AS second_highest
    FROM employees
    WHERE salary < (SELECT MAX(salary) FROM employees)
"""))
print("\nwindow version:\n", q("""
    SELECT salary AS second_highest FROM (
        SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
        FROM employees
    ) WHERE rnk = 2
"""))


subquery version:
    second_highest
0         88000.0



window version:
    second_highest
0         88000.0


In [74]:
# Q2 — highest salary per department
print(q("""
    SELECT d.dept_name, MAX(e.salary) AS max_salary
    FROM employees e
    JOIN departments d ON d.dept_id = e.dept_id
    GROUP BY d.dept_name
    ORDER BY max_salary DESC
"""))
# ...and WHO earns it (the classic follow-up):
print("\nemployees who earn their dept max:\n", q("""
    SELECT e.emp_name, d.dept_name, e.salary
    FROM employees e
    JOIN departments d ON d.dept_id = e.dept_id
    WHERE e.salary = (SELECT MAX(salary) FROM employees x WHERE x.dept_id = e.dept_id)
    ORDER BY e.salary DESC
"""))


   dept_name  max_salary
0         IT     90000.0
1    Finance     88000.0
2      Sales     66000.0
3         HR     62000.0
4  Marketing     60000.0
5    Support     48000.0



employees who earn their dept max:
   emp_name  dept_name   salary
0    Rahul         IT  90000.0
1    Rohan         IT  90000.0
2   Vikram    Finance  88000.0
3    Arjun      Sales  66000.0
4     Amit         HR  62000.0
5    Divya  Marketing  60000.0
6   Manish    Support  48000.0


In [75]:
# Q3 — employees earning above the company average
print(q("""
    SELECT emp_name, salary
    FROM employees
    WHERE salary > (SELECT AVG(salary) FROM employees)
    ORDER BY salary DESC
"""))


  emp_name   salary
0    Rahul  90000.0
1    Rohan  90000.0
2   Vikram  88000.0
3    Priya  75000.0
4     Neha  70000.0


In [76]:
# Q4 — delete duplicate rows, keep one copy
con.executescript("""
CREATE TABLE contacts(cid INTEGER PRIMARY KEY, name TEXT, email TEXT);
INSERT INTO contacts VALUES
 (1,'Ravi','r@x.com'), (2,'Ravi','r@x.com'), (3,'Sita','s@x.com'),
 (4,'Amit','a@x.com'), (5,'Sita','s@x.com');
""")
print("before:", len(q("SELECT * FROM contacts")), "rows")
con.execute("""
    DELETE FROM contacts
    WHERE cid IN (
        SELECT cid FROM (
            SELECT cid, ROW_NUMBER() OVER (
                       PARTITION BY name, email ORDER BY cid) AS rn
            FROM contacts
        ) WHERE rn > 1
    )
""")
print("after :", q("SELECT * FROM contacts"))


before: 5 rows
after :    cid  name    email
0    1  Ravi  r@x.com
1    3  Sita  s@x.com
2    4  Amit  a@x.com


In [77]:
# Q5 — running total of order amounts
print(q("""
    SELECT order_id, order_date, amount,
           SUM(amount) OVER (ORDER BY order_date, order_id
                 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
    FROM orders
    ORDER BY order_date, order_id
"""))


    order_id  order_date  amount  running_total
0          1  2024-01-05  1200.0         1200.0
1          2  2024-01-06   800.0         2000.0
2          4  2024-01-06  2200.0         4200.0
3          3  2024-01-20  1500.0         5700.0
4          5  2024-02-03   900.0         6600.0
5          6  2024-02-05  3000.0         9600.0
6          7  2024-02-06   450.0        10050.0
7          8  2024-02-07   700.0        10750.0
8          9  2024-02-14  1800.0        12550.0
9         10  2024-02-14   600.0        13150.0
10        11  2024-03-02  2500.0        15650.0
11        12  2024-03-08  1100.0        16750.0
12        13  2024-03-15   950.0        17700.0
13        14  2024-03-16  1300.0        19000.0
14        15  2024-03-21  2000.0        21000.0


In [78]:
# Q6 — month-over-month revenue growth (%)
print(q("""
    WITH monthly AS (
        SELECT strftime('%Y-%m', order_date) AS ym, SUM(amount) AS revenue
        FROM orders GROUP BY ym
    )
    SELECT ym, revenue,
           LAG(revenue) OVER (ORDER BY ym) AS prev_revenue,
           ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY ym))
                 / LAG(revenue) OVER (ORDER BY ym), 1) AS pct_growth
    FROM monthly
    ORDER BY ym
"""))


        ym  revenue  prev_revenue  pct_growth
0  2024-01   5700.0           NaN         NaN
1  2024-02   7450.0        5700.0        30.7
2  2024-03   7850.0        7450.0         5.4


In [79]:
# Q7 — customers who never placed an order (3 ways to the same answer)
print("LEFT JOIN + IS NULL:\n", q("""
    SELECT c.cust_name
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE o.order_id IS NULL
"""))
print("\nNOT EXISTS:\n", q("""
    SELECT c.cust_name FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
"""))


LEFT JOIN + IS NULL:
   cust_name
0      Tara
1     Imran

NOT EXISTS:
   cust_name
0      Tara
1     Imran


In [80]:
# Q8 — count of orders per day (and days with more than one order)
print(q("""
    SELECT order_date, COUNT(*) AS orders, SUM(amount) AS revenue
    FROM orders
    GROUP BY order_date
    ORDER BY orders DESC, order_date
"""))


    order_date  orders  revenue
0   2024-01-06       2   3000.0
1   2024-02-14       2   2400.0
2   2024-01-05       1   1200.0
3   2024-01-20       1   1500.0
4   2024-02-03       1    900.0
5   2024-02-05       1   3000.0
6   2024-02-06       1    450.0
7   2024-02-07       1    700.0
8   2024-03-02       1   2500.0
9   2024-03-08       1   1100.0
10  2024-03-15       1    950.0
11  2024-03-16       1   1300.0
12  2024-03-21       1   2000.0


In [81]:
# Q9 — consecutive-day activity streaks (LAG + date difference)
print("orders placed on the previous day too:\n", q("""
    SELECT customer_id, order_date, prev_date,
           CAST(julianday(order_date) - julianday(prev_date) AS INT) AS gap_days
    FROM (
        SELECT customer_id, order_date,
               LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_date
        FROM orders
        WHERE status = 'completed'
    )
    WHERE prev_date IS NOT NULL
      AND julianday(order_date) - julianday(prev_date) = 1
    ORDER BY customer_id, order_date
"""))


orders placed on the previous day too:
    customer_id  order_date   prev_date  gap_days
0            1  2024-01-06  2024-01-05         1
1            3  2024-02-06  2024-02-05         1
2            7  2024-03-16  2024-03-15         1


In [82]:
# Q10 — Nth highest salary *per department* (N = 2) with ROW_NUMBER
N = 2
print(f"2nd highest salary in each department:\n", q("""
    SELECT dept_id, emp_name, salary FROM (
        SELECT emp_name, dept_id, salary,
               ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn
        FROM employees
    ) WHERE rn = 2
    ORDER BY dept_id
"""))


2nd highest salary in each department:
    dept_id emp_name   salary
0        1    Sneha  58000.0
1        2    Rohan  90000.0
2        3     Neha  70000.0
3        4    Kavya  54000.0
4        6    Pooja  46000.0


### Interview quick-recap

| Question | One-line answer |
|---|---|
| 2nd highest salary | `MAX(salary) WHERE salary < MAX(...)` or `DENSE_RANK() = 2` |
| dept max + who | `GROUP BY` for the value, correlated `WHERE salary = (SELECT MAX ...)` for the name |
| above average | `WHERE salary > (SELECT AVG(salary) FROM t)` |
| delete dupes | `ROW_NUMBER() OVER (PARTITION BY keys)` → `DELETE rn > 1` |
| running total | `SUM() OVER (ORDER BY d ROWS UNBOUNDED PRECEDING→CURRENT)` |
| MoM growth | aggregate by month + `LAG()` |
| customers w/o orders | `LEFT JOIN ... WHERE o.id IS NULL` (or `NOT EXISTS`) |
| count per day | `GROUP BY date_col` |
| streaks | `LAG(date)` + `julianday(a)-julianday(b) = 1` |
| Nth per group | `ROW_NUMBER() OVER (PARTITION BY grp ORDER BY val DESC) = N` |

> **Gotcha:** `WHERE` filters rows, `HAVING` filters groups, `ORDER BY` runs
> last — an alias from `SELECT` can only be reused in `ORDER BY` (and in some
> engines, `HAVING` via the underlying expression).

---
## SQL cheat sheet

| Task | Query template |
|---|---|
| peek at a table | `SELECT * FROM t LIMIT 10;` |
| column list | `PRAGMA table_info(t);` |
| filter | `SELECT ... FROM t WHERE cond AND cond;` |
| pattern | `WHERE name LIKE 'A%';` |
| null-safe | `WHERE col IS NOT NULL;` / `COALESCE(col, default)` |
| sort + page | `ORDER BY col DESC LIMIT 10 OFFSET 20;` |
| categorise | `CASE WHEN x > 5 THEN 'a' ELSE 'b' END` |
| group counts | `SELECT g, COUNT(*), AVG(x) FROM t GROUP BY g HAVING COUNT(*) > 1;` |
| dedupe keys | `SELECT DISTINCT a, b FROM t;` |
| join | `FROM a JOIN b ON a.id = b.a_id` / `LEFT JOIN` to keep `a` |
| full outer (SQLite) | `LEFT JOIN ... UNION ALL ... WHERE right.id IS NULL` |
| subquery in FROM | `FROM (SELECT ...) AS d;` |
| reusable logic | `WITH cte AS (SELECT ...) SELECT ... FROM cte;` |
| hierarchy | `WITH RECURSIVE r AS (anchor UNION ALL step) SELECT ...;` |
| rank in group | `ROW_NUMBER()/RANK() OVER (PARTITION BY g ORDER BY v DESC)` |
| running total | `SUM(x) OVER (ORDER BY d ROWS UNBOUNDED PRECEDING AND CURRENT ROW)` |
| moving average | `AVG(x) OVER (ORDER BY d ROWS 2 PRECEDING AND CURRENT ROW)` |
| prev/next | `LAG(x, 1, default)` / `LEAD(x)` |
| top-N per group | outer `WHERE rn <= N` over `ROW_NUMBER() OVER (PARTITION BY g ...)` |
| MoM change | `GROUP BY strftime('%Y-%m', d)` + `LAG()` |
| set combine | `UNION` (distinct) / `UNION ALL` (keep dupes) / `INTERSECT` / `EXCEPT` |
| save a query | `CREATE VIEW v AS SELECT ...;` |
| speed up lookup | `CREATE INDEX i ON t(col);` then `EXPLAIN QUERY PLAN` |
| atomic change | `BEGIN; ...; COMMIT;` (`ROLLBACK` / `SAVEPOINT` to undo) |
| dedupe rows | `DELETE WHERE cid IN (SELECT ... WHERE rn > 1)` |
| stddev (SQLite) | `sqrt(AVG(x*x) - AVG(x)*AVG(x))` |
| pearson r | `(AVG(xy)-AVG(x)*AVG(y)) / (sd(x)*sd(y))` |

> **Interview short-list:** `WHERE` vs `HAVING` · execution order ·
> `COUNT(*)` vs `COUNT(col)` · NULL three-valued logic (`= NULL` never true) ·
> inner vs left vs full outer · correlated subquery runs per row ·
> `ROW_NUMBER` vs `RANK` vs `DENSE_RANK` · explicit frame for `LAST_VALUE` ·
> `UNION` vs `UNION ALL` · 1NF/2NF/3NF · ACID & isolation anomalies.

---
**Next notebook:** `08_statistics_probability.ipynb` — the maths behind the
numbers you just queried (distributions, tests, sampling).

**Playlist:** https://www.youtube.com/playlist?list=PLKnIA16_RmvbAlyx4_rdtR66B7EHX5k3z